# Lab 8, optional part: PyTorch

This notebook introduces PyTorch through the algorithms of Lab 8. The first three sections
cover tensors, automatic differentiation and the training loop; the fourth trains a neural
policy with REINFORCE on the cart-pole, and the fifth a deep Q-network (DQN) on the falling
blocks.

Replace the lines `...  # TODO` with your code.

This version runs without installation, for instance on Google Colab (File > Upload
notebook), where PyTorch is already installed. The cells of the section "Framework" below
write the course package `seqlearn` next to the notebook: run them first.

## Framework

The cells of this section write the course package `seqlearn` next to the notebook. Run them, without modifying them, before the rest of the notebook.

In [ ]:
import os

os.makedirs("seqlearn", exist_ok=True)

In [ ]:
%%writefile seqlearn/bandits.py
"""Stochastic bandits: environments, agent interface, interaction loop."""

import numpy as np


class Bandit:
    """Stochastic K-armed bandit with arm means `means`."""

    def __init__(self, means, rng=None):
        self.means = np.asarray(means, dtype=float)
        self.n_arms = len(self.means)
        self.rng = np.random.default_rng(rng)

    @property
    def best_mean(self):
        return self.means.max()

    @property
    def gaps(self):
        return self.best_mean - self.means

    def pull(self, arm):
        """Pull arm `arm` and return the reward."""
        raise NotImplementedError


class BernoulliBandit(Bandit):
    def pull(self, arm):
        return float(self.rng.random() < self.means[arm])


class GaussianBandit(Bandit):
    def __init__(self, means, sigma=1.0, rng=None):
        super().__init__(means, rng)
        self.sigma = sigma

    def pull(self, arm):
        return self.means[arm] + self.sigma * self.rng.standard_normal()


class Agent:
    """Bandit agent. Subclasses implement `act`.

    `update` keeps track of `t` (number of rounds played), `counts[a]` (number of pulls
    of arm a) and `mu_hat[a]` (empirical mean of arm a, zero until the arm is pulled),
    using the incremental update seen in the lecture.
    """

    def __init__(self, n_arms, rng=None):
        self.n_arms = n_arms
        self.rng = np.random.default_rng(rng)
        self.reset()

    def reset(self):
        self.t = 0
        self.counts = np.zeros(self.n_arms, dtype=int)
        self.mu_hat = np.zeros(self.n_arms)

    def act(self):
        """Return the arm to play at the next round."""
        raise NotImplementedError

    def update(self, arm, reward):
        self.t += 1
        self.counts[arm] += 1
        self.mu_hat[arm] += (reward - self.mu_hat[arm]) / self.counts[arm]


class UniformAgent(Agent):
    def act(self):
        return int(self.rng.integers(self.n_arms))


def run(env, agent, horizon):
    """Let `agent` play on `env` for `horizon` rounds.

    Return two arrays of length `horizon`: arms played and rewards.
    """
    agent.reset()
    arms = np.empty(horizon, dtype=int)
    rewards = np.empty(horizon)
    for t in range(horizon):
        arm = agent.act()
        if not (isinstance(arm, (int, np.integer)) and 0 <= arm < env.n_arms):
            raise ValueError(
                f"act() must return an integer between 0 and {env.n_arms - 1}, "
                f"but returned {arm!r} at round {t + 1}."
            )
        reward = env.pull(arm)
        agent.update(arm, reward)
        arms[t], rewards[t] = arm, reward
    return arms, rewards


def run_many(make_env, make_agent, horizon, n_runs, seed=0):
    """Repeat `run` over `n_runs` independent and reproducible runs.

    `make_env(rng)` builds the environment and `make_agent(n_arms, rng)` the agent. Each
    run gets its own generators, derived from `seed`: two calls with the same seed see the
    same environment randomness, whatever the agent.
    Return two arrays of shape (n_runs, horizon): arms played and rewards.
    """
    arms = np.empty((n_runs, horizon), dtype=int)
    rewards = np.empty((n_runs, horizon))
    for i, child in enumerate(np.random.SeedSequence(seed).spawn(n_runs)):
        env_seed, agent_seed = child.spawn(2)
        env = make_env(np.random.default_rng(env_seed))
        agent = make_agent(env.n_arms, np.random.default_rng(agent_seed))
        arms[i], rewards[i] = run(env, agent, horizon)
    return arms, rewards


def pseudo_regret(arms, gaps):
    """Cumulative pseudo-regret, sum of gaps[A_s] for s <= t, along the last axis."""
    return np.cumsum(np.asarray(gaps)[arms], axis=-1)

In [ ]:
%%writefile seqlearn/agents.py
"""Reference agents from the previous labs, used as baselines."""

import numpy as np

from .bandits import Agent


class UCBAgent(Agent):
    """UCB(alpha): index mu_hat + sqrt(alpha log t / N). alpha = 2 gives UCB1 (Labs 1 and 2)."""

    def __init__(self, n_arms, alpha=2.0, rng=None):
        self.alpha = alpha
        super().__init__(n_arms, rng)

    def index(self):
        return self.mu_hat + np.sqrt(self.alpha * np.log(self.t + 1) / self.counts)

    def act(self):
        if self.t < self.n_arms:
            return self.t
        return int(np.argmax(self.index()))

In [ ]:
%%writefile seqlearn/linear.py
"""Linear bandits: environments, agent interface, interaction loop."""

import numpy as np


class LinearBandit:
    """Linear bandit with Bernoulli rewards.

    At each round, arm a has a feature vector x_a in R^d, given by `features()` as an
    array of shape (n_arms, dim). Pulling arm a pays 1 with probability <theta, x_a> and 0
    otherwise. The parameter `theta` is unknown to the agent.
    """

    def __init__(self, theta, rng=None):
        self.theta = np.asarray(theta, dtype=float)
        self.dim = len(self.theta)
        self.rng = np.random.default_rng(rng)

    def features(self):
        """Feature vectors of the arms at the current round, of shape (n_arms, dim)."""
        raise NotImplementedError

    def means(self):
        """Mean rewards of the arms at the current round."""
        return self.features() @ self.theta

    def pull(self, arm):
        """Pull arm `arm`, return the reward and move to the next round."""
        reward = float(self.rng.random() < self.means()[arm])
        self._next_round()
        return reward

    def _next_round(self):
        pass


class CatalogBandit(LinearBandit):
    """Fixed catalog: item a has the feature vector items[a], the same at every round."""

    def __init__(self, items, theta, rng=None):
        super().__init__(theta, rng)
        self.items = np.asarray(items, dtype=float)
        self.n_arms = len(self.items)

    def features(self):
        return self.items


class RecommendationBandit(LinearBandit):
    """Recommendation with user contexts.

    `appeal[a, j]` is the probability that item a is clicked by a user in mood j. At each
    round a user arrives with a profile `context`, a probability vector over the moods, and
    item a is clicked with probability <context, appeal[a]>. This is a linear bandit of
    dimension n_arms * n_moods: the feature vector of item a is the profile written in
    block a, and zero elsewhere, and theta is the matrix `appeal` flattened.
    """

    def __init__(self, appeal, concentration=0.3, rng=None):
        self.appeal = np.asarray(appeal, dtype=float)
        self.n_arms, self.n_moods = self.appeal.shape
        self.concentration = concentration
        super().__init__(self.appeal.ravel(), rng)
        self._next_round()

    def _next_round(self):
        self.context = self.rng.dirichlet(self.concentration * np.ones(self.n_moods))
        self._features = np.kron(np.eye(self.n_arms), self.context)

    def features(self):
        return self._features


def make_catalog(n_items, dim, rng=None):
    """Random catalog: `n_items` unit feature vectors with nonnegative entries, and a unit
    parameter theta with nonnegative entries, so that all mean rewards lie in [0, 1]."""
    rng = np.random.default_rng(rng)
    items = np.abs(rng.standard_normal((n_items, dim)))
    items /= np.linalg.norm(items, axis=1, keepdims=True)
    theta = np.abs(rng.standard_normal(dim))
    theta /= np.linalg.norm(theta)
    return items, theta


class LinearAgent:
    """Linear bandit agent. Subclasses implement `act` and `update`."""

    def __init__(self, dim, rng=None):
        self.dim = dim
        self.rng = np.random.default_rng(rng)
        self.reset()

    def reset(self):
        self.t = 0

    def act(self, features):
        """Return the arm to play, given the array `features` of shape (n_arms, dim)."""
        raise NotImplementedError

    def update(self, x, arm, reward):
        """Take into account the reward of arm `arm`, whose feature vector is `x`."""
        self.t += 1


class ContextFree(LinearAgent):
    """Wrap a K-armed agent (Labs 1 and 2) so that it plays a linear bandit, ignoring the
    features: each arm index is treated as an unrelated arm."""

    def __init__(self, agent):
        self.agent = agent
        super().__init__(dim=None, rng=agent.rng)

    def reset(self):
        super().reset()
        self.agent.reset()

    def act(self, features):
        return self.agent.act()

    def update(self, x, arm, reward):
        super().update(x, arm, reward)
        self.agent.update(arm, reward)


def run_linear(env, agent, horizon):
    """Let `agent` play on the linear bandit `env` for `horizon` rounds.

    Return three arrays of length `horizon`: arms played, rewards, and instantaneous
    regrets max_a <theta, x_a> - <theta, x_{A_t}>.
    """
    agent.reset()
    arms = np.empty(horizon, dtype=int)
    rewards = np.empty(horizon)
    regrets = np.empty(horizon)
    for t in range(horizon):
        features = env.features()
        arm = agent.act(features)
        if not (isinstance(arm, (int, np.integer)) and 0 <= arm < len(features)):
            raise ValueError(
                f"act() must return an integer between 0 and {len(features) - 1}, "
                f"but returned {arm!r} at round {t + 1}."
            )
        means = features @ env.theta
        reward = env.pull(arm)
        agent.update(features[arm], arm, reward)
        arms[t], rewards[t], regrets[t] = arm, reward, means.max() - means[arm]
    return arms, rewards, regrets


def run_many_linear(make_env, make_agent, horizon, n_runs, seed=0):
    """Repeat `run_linear` over `n_runs` independent and reproducible runs.

    `make_env(rng)` builds the environment and `make_agent(env, rng)` the agent.
    Return the cumulative pseudo-regret, of shape (n_runs, horizon).
    """
    regret = np.empty((n_runs, horizon))
    for i, child in enumerate(np.random.SeedSequence(seed).spawn(n_runs)):
        env_seed, agent_seed = child.spawn(2)
        env = make_env(np.random.default_rng(env_seed))
        agent = make_agent(env, np.random.default_rng(agent_seed))
        regret[i] = np.cumsum(run_linear(env, agent, horizon)[2])
    return regret

In [ ]:
%%writefile seqlearn/adversarial.py
"""Adversarial sequences of rewards: full information (experts) and bandit feedback."""

import numpy as np


class SequenceBandit:
    """Bandit whose rewards are fixed in advance: pulling arm a at round t pays rewards[t, a].

    No randomness and no means: the sequence can be arbitrary, for instance built against a
    given strategy. The regret is measured against the best fixed arm in hindsight.
    """

    def __init__(self, rewards):
        self.rewards = np.asarray(rewards, dtype=float)
        self.horizon, self.n_arms = self.rewards.shape
        self.t = 0

    def pull(self, arm):
        reward = self.rewards[self.t, arm]
        self.t += 1
        return reward


def alternating_gains(horizon):
    """Gains of two experts: (1/2, 0) at the first round, then (0, 1), (1, 0), (0, 1), ..."""
    gains = np.zeros((horizon, 2))
    gains[0] = [0.5, 0.0]
    gains[1::2, 1] = 1.0
    gains[2::2, 0] = 1.0
    return gains


def stochastic_gains(means, horizon, rng=None):
    """Gains of experts drawn independently from Bernoulli distributions with means `means`."""
    rng = np.random.default_rng(rng)
    return (rng.random((horizon, len(means))) < np.asarray(means)).astype(float)


def regret_vs_best_arm(rewards, received):
    """Cumulative regret against the best fixed arm in hindsight, at each round.

    `rewards` has shape (horizon, n_arms) and `received` (horizon,) or (n_runs, horizon): the
    gains collected by the strategy. The regret at round t is the cumulative reward of the
    best arm over rounds 1..t minus the cumulative gain of the strategy.
    """
    best = np.cumsum(rewards, axis=0).max(axis=1)
    return best - np.cumsum(received, axis=-1)

In [ ]:
%%writefile seqlearn/offpolicy.py
"""Off-policy evaluation on the recommendation problem with user profiles (Lab 5).

At each round a user arrives with a profile, a probability vector over N_MOODS moods, and
item a is clicked with probability <profile, APPEAL[a]>. A policy is a function that maps
an array of profiles, of shape (n, N_MOODS), to the array of the probabilities of the
items, of shape (n, N_ITEMS): row i is the distribution of the item recommended to user i.
"""

import numpy as np

APPEAL = np.array([[0.8, 0.2, 0.3, 0.1],
                   [0.2, 0.8, 0.2, 0.3],
                   [0.3, 0.2, 0.8, 0.2],
                   [0.1, 0.3, 0.2, 0.8],
                   [0.5, 0.5, 0.5, 0.5]])
N_ITEMS, N_MOODS = APPEAL.shape
CONCENTRATION = 0.3

# Click rates estimated by the recommender currently in production: noisy, and biased
# towards the last item.
_OLD_APPEAL = APPEAL + np.random.default_rng(1).normal(0, 0.15, APPEAL.shape)
_OLD_APPEAL[-1] += 0.2


def draw_profiles(n, rng=None):
    """Profiles of `n` users, of shape (n, N_MOODS)."""
    return np.random.default_rng(rng).dirichlet(CONCENTRATION * np.ones(N_MOODS), size=n)


def click_probabilities(profiles):
    """Click probability of each item for each user, of shape (n, N_ITEMS)."""
    return profiles @ APPEAL.T


def softmax(scores, temperature=1.0):
    """Softmax of each row of `scores`."""
    z = np.asarray(scores, dtype=float) / temperature
    z = np.exp(z - z.max(axis=-1, keepdims=True))
    return z / z.sum(axis=-1, keepdims=True)


def old_recommender(profiles):
    """Policy in production: softmax of its estimated click rates, temperature 0.1."""
    return softmax(profiles @ _OLD_APPEAL.T, temperature=0.1)


def uniform_policy(profiles):
    return np.full((len(profiles), N_ITEMS), 1 / N_ITEMS)


def constant_policy(item):
    """Policy that always recommends `item`."""
    def policy(profiles):
        probs = np.zeros((len(profiles), N_ITEMS))
        probs[:, item] = 1.0
        return probs
    return policy


def collect_logs(policy, n, rng=None):
    """Let `policy` recommend to `n` users and record what happens.

    Return a dictionary of arrays of length n: "profiles" (shape (n, N_MOODS)), "items" (the
    recommended items), "clicks" (0 or 1) and "propensities" (the probability with which the
    policy chose the recommended item).
    """
    rng = np.random.default_rng(rng)
    profiles = draw_profiles(n, rng)
    probs = policy(profiles)
    items = (probs.cumsum(axis=1) < rng.random((n, 1))).sum(axis=1).clip(max=N_ITEMS - 1)
    rows = np.arange(n)
    clicks = (rng.random(n) < click_probabilities(profiles)[rows, items]).astype(float)
    return {"profiles": profiles, "items": items, "clicks": clicks, "propensities": probs[rows, items]}


def true_value(policy, n=200_000, rng=12345):
    """Click rate of `policy`, by exact averaging over the items on `n` simulated users."""
    profiles = draw_profiles(n, rng)
    return float(np.mean(np.sum(policy(profiles) * click_probabilities(profiles), axis=1)))

In [ ]:
%%writefile seqlearn/preferences.py
"""Duels between models (Lab 6): Bradley-Terry arena and ranking utilities."""

import numpy as np


def sigmoid(z):
    return 1 / (1 + np.exp(-np.asarray(z, dtype=float)))


class Arena:
    """Duels between `len(scores)` models: model i beats model j with probability
    sigmoid(scores[i] - scores[j]) (Bradley-Terry model). The scores are unknown to the agent.
    """

    def __init__(self, scores, rng=None):
        self.scores = np.asarray(scores, dtype=float)
        self.n_models = len(self.scores)
        self.rng = np.random.default_rng(rng)
        self.n_duels = 0

    @property
    def best(self):
        return int(np.argmax(self.scores))

    def duel(self, i, j):
        """Make models i and j compete once: return 1.0 if i wins and 0.0 otherwise."""
        if i == j:
            raise ValueError("A model cannot duel itself.")
        self.n_duels += 1
        return float(self.rng.random() < sigmoid(self.scores[i] - self.scores[j]))


def make_arena(n_models=8, rng=None):
    """Arena with scores drawn from a standard Gaussian distribution, in random order."""
    rng = np.random.default_rng(rng)
    return Arena(rng.normal(0, 1, n_models), rng)


def record(wins, i, j, i_wins):
    """Add the result of one duel to the matrix `wins`, where wins[i, j] counts the wins of i
    against j."""
    if i_wins:
        wins[i, j] += 1
    else:
        wins[j, i] += 1


def kendall_tau(x, y):
    """Kendall rank correlation between two score vectors: the proportion of concordant pairs
    minus the proportion of discordant pairs, in [-1, 1]."""
    x, y = np.asarray(x), np.asarray(y)
    i, j = np.triu_indices(len(x), k=1)
    return float(np.mean(np.sign(x[i] - x[j]) * np.sign(y[i] - y[j])))

In [ ]:
%%writefile seqlearn/mdp.py
"""Finite Markov decision processes and grid worlds (Lab 7)."""

import matplotlib.pyplot as plt
import numpy as np

from .plotting import MUTED, TEXT

# Actions of the grid worlds, as (row, column) moves.
ACTIONS = ["up", "right", "down", "left"]
MOVES = [(-1, 0), (0, 1), (1, 0), (0, -1)]


class TabularMDP:
    """Finite MDP given by its transition probabilities and expected rewards.

    P[s, a, s'] is the probability of moving to s' when playing a in s, and R[s, a] the
    expected reward of this step. Terminal states are absorbing, with reward zero. `reset`
    and `step` simulate the MDP, for algorithms that do not use P and R.
    """

    def __init__(self, P, R, start, terminal=(), rng=None):
        self.P = np.asarray(P, dtype=float)
        self.R = np.asarray(R, dtype=float)
        self.n_states, self.n_actions = self.R.shape
        self.start = start
        self.terminal = np.zeros(self.n_states, dtype=bool)
        self.terminal[list(terminal)] = True
        self.rng = np.random.default_rng(rng)
        self._rewards = {}  # (s, a, s') -> reward, when it depends on s'
        self.state = start

    def reset(self):
        self.state = self.start
        return self.state

    def step(self, action):
        """Play `action` in the current state: return (next state, reward, done)."""
        s = self.state
        s_next = int(self.rng.choice(self.n_states, p=self.P[s, action]))
        reward = self._rewards.get((s, action, s_next), self.R[s, action])
        self.state = s_next
        return s_next, reward, bool(self.terminal[s_next])


class GridWorld(TabularMDP):
    """Grid world built from a list of strings, one per row.

    Cells: '.' free, '#' wall, 'S' start, 'G' goal (reward `goal`, terminal), 'T' trap
    (reward `trap`, terminal), 'g' small goal (reward `small_goal`, terminal), 'C' cliff
    (reward `cliff` and back to the start). Every move costs `step`. With probability
    `slip`, the move goes in a uniformly random direction instead. Moving into a wall or
    out of the grid leaves the agent in place.
    """

    def __init__(self, layout, step=0.0, goal=1.0, trap=-1.0, small_goal=0.1, cliff=-100.0, slip=0.0,
                 rng=None):
        self.layout = [list(row) for row in layout]
        self.n_rows, self.n_cols = len(layout), len(layout[0])
        self.cells = [(i, j) for i in range(self.n_rows) for j in range(self.n_cols) if layout[i][j] != "#"]
        index = {cell: k for k, cell in enumerate(self.cells)}
        self.index = index
        n = len(self.cells)
        start = next(index[c] for c in self.cells if self.kind(c) == "S")
        exit_reward = {"G": goal, "T": trap, "g": small_goal}
        terminal = [index[c] for c in self.cells if self.kind(c) in exit_reward]
        P = np.zeros((n, 4, n))
        R = np.zeros((n, 4))
        rewards = {}
        for c in self.cells:
            s = index[c]
            if s in terminal:
                P[s, :, s] = 1.0
                continue
            for a in range(4):
                outcomes = [(1 - slip, a)] + [(slip / 4, b) for b in range(4)]
                for prob, b in outcomes:
                    if prob == 0:
                        continue
                    target = (c[0] + MOVES[b][0], c[1] + MOVES[b][1])
                    if target not in index:
                        target = c
                    r = step
                    kind = self.kind(target)
                    if kind == "C":
                        r, target = step + cliff, self.cells[start]
                    elif kind in exit_reward:
                        r = step + exit_reward[kind]
                    t = index[target]
                    P[s, a, t] += prob
                    R[s, a] += prob * r
                    rewards[(s, a, t)] = r
        super().__init__(P, R, start, terminal, rng)
        self._rewards = rewards

    def kind(self, cell):
        return self.layout[cell[0]][cell[1]]

    def plot(self, ax, values=None, policy=None, title=None):
        """Draw the grid, with `values` (one per state) as colors and `policy` (one action per
        state) as arrows."""
        grid = np.full((self.n_rows, self.n_cols), np.nan)
        if values is not None:
            for k, c in enumerate(self.cells):
                grid[c] = values[k]
            image = ax.imshow(grid, cmap="Blues")
            plt.colorbar(image, ax=ax, fraction=0.035)
        for i in range(self.n_rows):
            for j in range(self.n_cols):
                kind = self.layout[i][j]
                if kind == "#":
                    ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, color=TEXT))
                elif kind == "C":
                    ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, color="#e34948", alpha=0.6))
                if kind in "SGTgC":
                    ax.text(j, i, kind, ha="center", va="center", fontsize=10, fontweight="bold", color=TEXT)
        if policy is not None:
            for k, c in enumerate(self.cells):
                if self.terminal[k] or self.kind(c) == "C":
                    continue
                di, dj = MOVES[int(policy[k])]
                ax.annotate("", xy=(c[1] + 0.3 * dj, c[0] + 0.3 * di), xytext=(c[1] - 0.3 * dj, c[0] - 0.3 * di),
                            arrowprops=dict(arrowstyle="->", color="black", lw=1.2))
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_xlim(-0.5, self.n_cols - 0.5)
        ax.set_ylim(self.n_rows - 0.5, -0.5)
        for spine in ax.spines.values():
            spine.set_color(MUTED)
        if title:
            ax.set_title(title)


MAZE = [
    "S...#....",
    ".##.#.##.",
    ".#.....#.",
    ".#.##T.#.",
    "...#...#G",
]

CHOICE = [
    "..........",
    ".g.......G",
    "..........",
    "S.........",
]

CLIFF = [
    "............",
    "............",
    "............",
    "SCCCCCCCCCCG",
]


def make_maze(slip=0.1, rng=None):
    """Maze of Lab 7: goal G (+1) and trap T (-1), slippery moves."""
    return GridWorld(MAZE, step=0.0, goal=1.0, trap=-1.0, slip=slip, rng=rng)


def make_choice(rng=None):
    """A small reward g (0.1) close to the start and a large one G (1) far away."""
    return GridWorld(CHOICE, step=0.0, goal=1.0, small_goal=0.1, rng=rng)


def make_cliff(rng=None):
    """Cliff walking (Sutton and Barto, example 6.6): -1 per move, -100 and back to the start
    when falling from the cliff C, episode ends at G."""
    return GridWorld(CLIFF, step=-1.0, goal=0.0, cliff=-100.0, rng=rng)

In [ ]:
%%writefile seqlearn/blocks.py
"""Falling blocks: a small stacking game for approximate Q-learning (Lab 8)."""

import matplotlib.pyplot as plt
import numpy as np

from .plotting import COLORS, MUTED, TEXT

# Default pieces, drawn row by row from the top with '#' for a filled cell. Any list of such
# drawings can be passed to Blocks(pieces=...).
PIECES = [
    ["#"],
    ["##"],
    ["###"],
    ["#.",
     "##"],
]


def _cells(drawing):
    """Cells (row from the bottom, column) of a drawing."""
    height = len(drawing)
    return [(height - 1 - i, j) for i, row in enumerate(drawing) for j, c in enumerate(row) if c == "#"]


def _normalize(cells):
    r0, c0 = min(r for r, _ in cells), min(c for _, c in cells)
    return tuple(sorted((r - r0, c - c0) for r, c in cells))


def _rotations(drawing):
    """Distinct rotations of a piece, each as a sorted tuple of cells."""
    shapes, cells = [], _normalize(_cells(drawing))
    for _ in range(4):
        if cells not in shapes:
            shapes.append(cells)
        cells = _normalize([(c, -r) for r, c in cells])  # quarter turn
    return shapes


class Blocks:
    """Pieces fall one at a time in a well of `width` columns and `height` rows.

    The state is (board, piece): `board` is a boolean array of shape (height, width), row 0 at
    the bottom, and `piece` the index of the falling piece in `pieces`. An action is a
    placement (rotation, column): the piece is turned, moved to the column and dropped
    straight down. Each full row is removed and gives a reward 1. The game ends when a
    piece sticks out of the top of the well, with a reward `lose`, or after `max_pieces`
    pieces. `rows` counts the rows removed since the start of the game.
    """

    def __init__(self, width=6, height=10, pieces=PIECES, max_pieces=500, lose=-1.0, rng=None):
        self.width, self.height, self.max_pieces, self.lose = width, height, max_pieces, lose
        self.pieces = [_rotations(p) for p in pieces]
        # Placements of each piece: (rotation, leftmost column) inside the well
        self.placements = [[(k, c) for k, shape in enumerate(rotations)
                            for c in range(width - max(j for _, j in shape))]
                           for rotations in self.pieces]
        self.rng = np.random.default_rng(rng)

    def reset(self):
        self.board = np.zeros((self.height, self.width), dtype=bool)
        self.piece = int(self.rng.integers(len(self.pieces)))
        self.count = self.rows = 0
        return self.board.copy(), self.piece

    def actions(self, piece=None):
        """Legal placements (rotation, column) of `piece` (by default the falling one)."""
        return self.placements[self.piece if piece is None else piece]

    def drop(self, board, piece, action):
        """Board after dropping `piece` with `action`, before removing full rows, or None if the
        piece sticks out of the top. Does not change the game."""
        shape = self.pieces[piece][action[0]]
        column = action[1]
        filled = board.any(axis=0)
        tops = np.where(filled, self.height - np.argmax(board[::-1], axis=0), 0)
        bottom = {}
        for r, j in shape:
            bottom[j] = min(r, bottom.get(j, r))
        row = max(tops[column + j] - r for j, r in bottom.items())
        if row + max(r for r, _ in shape) >= self.height:
            return None
        new = board.copy()
        for r, j in shape:
            new[row + r, column + j] = True
        return new

    @staticmethod
    def clear(board):
        """Board after removing its full rows, and the number of rows removed."""
        full = board.all(axis=1)
        kept = board[~full]
        cleared = int(full.sum())
        return np.vstack([kept, np.zeros((cleared, board.shape[1]), dtype=bool)]), cleared

    def step(self, action):
        """Play the placement `action`: return ((board, piece), reward, done)."""
        dropped = self.drop(self.board, self.piece, action)
        self.count += 1
        if dropped is None:
            return (self.board.copy(), self.piece), self.lose, True
        self.board, cleared = self.clear(dropped)
        self.rows += cleared
        self.piece = int(self.rng.integers(len(self.pieces)))
        return (self.board.copy(), self.piece), float(cleared), self.count >= self.max_pieces

    def draw(self, ax, board=None, piece=None, title=None):
        """Draw a board (by default the current one) and, next to it, the falling piece."""
        board = self.board if board is None else board
        ax.add_patch(plt.Rectangle((-0.5, -0.5), self.width, self.height, fill=False, ec=MUTED, lw=1.5))
        for r, j in zip(*np.nonzero(board)):
            ax.add_patch(plt.Rectangle((j - 0.45, r - 0.45), 0.9, 0.9, color=COLORS[0]))
        if piece is not None:
            for r, j in self.pieces[piece][0]:
                ax.add_patch(plt.Rectangle((self.width + 0.55 + j, self.height - 3.45 + r), 0.9, 0.9,
                                           color=COLORS[1]))
            ax.text(self.width + 0.5, self.height - 0.8, "piece", color=TEXT, fontsize="small")
        ax.set_xlim(-0.7, self.width + (3.8 if piece is not None else 0.2))
        ax.set_ylim(-0.7, self.height - 0.3)
        ax.set_aspect("equal")
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(False)
        for spine in ax.spines.values():
            spine.set_visible(False)
        if title:
            ax.set_title(title, color=TEXT)


def board_features(boards):
    """Features of a board, or of each board of a stack of shape (n, height, width): column
    heights, number of holes (empty cells below a filled one), bumpiness (sum of the height
    differences of neighboring columns) and maximal height."""
    boards = np.asarray(boards, dtype=bool)
    height = boards.shape[-2]
    tops = np.where(boards.any(axis=-2), height - np.argmax(boards[..., ::-1, :], axis=-2), 0)
    holes = tops.sum(axis=-1) - boards.sum(axis=(-2, -1))
    bumpiness = np.abs(np.diff(tops, axis=-1)).sum(axis=-1)
    return np.concatenate([tops, holes[..., None], bumpiness[..., None], tops.max(axis=-1, keepdims=True)],
                          axis=-1).astype(float)


def placement_features(env, board, piece):
    """Features phi(s, a) of all the legal placements a of `piece` on `board`.

    Returns (actions, phi, rewards, lost): the list of the placements, the array phi of shape
    (n_placements, 11) whose rows are a constant 1, the features of the board obtained (after
    removing the full rows) divided by the height of the well, and the number of rows
    removed; the rewards; and a boolean array, True for the placements that end the game
    (their row of phi is zero, their reward is env.lose)."""
    actions = env.actions(piece)
    n = len(actions)
    boards = np.zeros((n, env.height, env.width), dtype=bool)
    lost = np.zeros(n, dtype=bool)
    for k, action in enumerate(actions):
        dropped = env.drop(board, piece, action)
        if dropped is None:
            lost[k] = True
        else:
            boards[k] = dropped
    full = boards.all(axis=2)
    cleared = full.sum(axis=1)
    for k in np.nonzero(cleared)[0]:
        boards[k] = env.clear(boards[k])[0]
    phi = np.concatenate([np.ones((n, 1)), board_features(boards) / env.height, cleared[:, None]], axis=1)
    phi[lost] = 0.0
    rewards = np.where(lost, env.lose, cleared.astype(float))
    return actions, phi, rewards, lost

In [ ]:
%%writefile seqlearn/control.py
"""Cart-pole balancing and softmax policies with linear features (Lab 8)."""

import matplotlib.pyplot as plt
import numpy as np

from .plotting import COLORS, MUTED, TEXT


class CartPole:
    """Cart-pole balancing (Barto, Sutton and Anderson, 1983), with the constants of Gymnasium.

    The state is (x, x_dot, theta, theta_dot): position and speed of the cart, angle of the
    pole (0 is upright) and its angular speed. Action 0 pushes the cart to the left, action 1
    to the right. Each step gives a reward 1, including the last one; the episode ends when
    |theta| > 12 degrees or |x| > 2.4 (failure), or after `max_steps` steps.
    """

    gravity, mass_cart, mass_pole, half_length, force, tau = 9.8, 1.0, 0.1, 0.5, 10.0, 0.02
    theta_max, x_max = 12 * np.pi / 180, 2.4
    n_actions, n_features = 2, 4

    def __init__(self, max_steps=500, rng=None):
        self.max_steps = max_steps
        self.rng = np.random.default_rng(rng)
        self.state = None

    def reset(self):
        self.state = self.rng.uniform(-0.05, 0.05, size=4)
        self.t = 0
        return self.state.copy()

    def step(self, action):
        """Play `action` (0 or 1): return (next state, reward, done)."""
        x, x_dot, theta, theta_dot = self.state
        force = self.force if action == 1 else -self.force
        cos, sin = np.cos(theta), np.sin(theta)
        total = self.mass_cart + self.mass_pole
        pole_ml = self.mass_pole * self.half_length
        temp = (force + pole_ml * theta_dot**2 * sin) / total
        theta_acc = (self.gravity * sin - cos * temp) / (
            self.half_length * (4.0 / 3.0 - self.mass_pole * cos**2 / total))
        x_acc = temp - pole_ml * theta_acc * cos / total
        x, x_dot = x + self.tau * x_dot, x_dot + self.tau * x_acc
        theta, theta_dot = theta + self.tau * theta_dot, theta_dot + self.tau * theta_acc
        self.state = np.array([x, x_dot, theta, theta_dot])
        self.t += 1
        failed = abs(x) > self.x_max or abs(theta) > self.theta_max
        return self.state.copy(), 1.0, bool(failed or self.t >= self.max_steps)

    def draw(self, ax, state, title=None):
        """Draw the cart and the pole in `state`."""
        x, theta = state[0], state[2]
        ax.plot([-self.x_max - 0.4, self.x_max + 0.4], [0, 0], color=MUTED, lw=1)
        for edge in (-self.x_max, self.x_max):
            ax.plot([edge, edge], [-0.05, 0.05], color=MUTED, lw=1)
        ax.add_patch(plt.Rectangle((x - 0.25, 0), 0.5, 0.25, color=COLORS[0]))
        top = (x + 2 * self.half_length * np.sin(theta), 0.25 + 2 * self.half_length * np.cos(theta))
        ax.plot([x, top[0]], [0.25, top[1]], color=COLORS[1], lw=4, solid_capstyle="round")
        ax.set_xlim(-self.x_max - 0.5, self.x_max + 0.5)
        ax.set_ylim(-0.2, 1.4)
        ax.set_aspect("equal")
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(False)
        for spine in ax.spines.values():
            spine.set_visible(False)
        if title:
            ax.set_title(title, color=TEXT)


# Typical magnitudes of the four coordinates of the state, used to scale the features.
SCALE = np.array([2.4, 3.0, 0.21, 3.0])


def quadratic_features(state):
    """Features (1, u_i, u_i u_j for i <= j) of a state, where u = state / SCALE, for value
    functions (15 features for the cart-pole)."""
    state = np.asarray(state, dtype=float) / SCALE
    i, j = np.triu_indices(len(state))
    return np.concatenate([[1.0], state, state[i] * state[j]])


def run_policy(env, act, n_episodes=1, rng=None):
    """Lengths of `n_episodes` episodes where the action in state s is `act(s, rng)`."""
    rng = np.random.default_rng(rng)
    lengths = []
    for _ in range(n_episodes):
        state, done, steps = env.reset(), False, 0
        while not done:
            state, _, done = env.step(act(state, rng))
            steps += 1
        lengths.append(steps)
    return np.array(lengths)

In [ ]:
%%writefile seqlearn/language.py
"""A character-level language model and simulated annotators, for RLHF and DPO (Lab 9).

The vocabulary has 27 tokens: the letters a to z (tokens 0 to 25) and the end of the word
(token 26). A word is generated letter by letter by a bigram model: the next token depends on
the previous letter only, or on the start of the word (row 26 of the tables). A bigram model
is given by a matrix of logits L of shape (27, 27): row p holds the logits of the next token
after the previous token p (p = 26 at the start), and the policy is softmax(L[p]).
"""

import re

import numpy as np

ALPHABET = "abcdefghijklmnopqrstuvwxyz"
END = 26  # token of the end of a word
START = 26  # row of the tables used for the first letter
MAX_LENGTH = 10

# Bigram counts of the 61171 words of 3 to 10 letters of the English word list of Debian
# (package wamerican): COUNTS[p, c] is the number of times token c follows p.
COUNTS = np.array([
    [44, 1210, 1971, 1523, 175, 366, 1141, 238, 1296, 45, 539, 3629, 1563, 5041, 52, 1073, 51, 4444, 2419, 4177, 752, 658, 455, 156, 757, 255, 1713],
    [1443, 405, 19, 34, 1525, 8, 4, 17, 927, 35, 2, 1392, 30, 26, 1158, 8, 1, 867, 317, 61, 831, 15, 11, 1, 137, 0, 169],
    [2361, 7, 265, 11, 1741, 3, 6, 2381, 1039, 0, 1623, 801, 10, 11, 2595, 7, 29, 1084, 228, 1007, 811, 4, 0, 0, 228, 9, 640],
    [944, 82, 43, 418, 3446, 45, 265, 66, 2307, 50, 5, 493, 91, 79, 1017, 37, 1, 601, 879, 36, 522, 61, 107, 0, 204, 9, 7057],
    [2348, 421, 1329, 6783, 1399, 627, 549, 165, 488, 55, 135, 2695, 1132, 4360, 323, 814, 133, 8922, 8819, 2097, 228, 658, 472, 677, 496, 111, 6272],
    [637, 12, 9, 5, 752, 625, 5, 4, 1081, 2, 3, 727, 6, 9, 790, 4, 0, 538, 134, 293, 623, 0, 4, 0, 151, 0, 201],
    [1056, 18, 2, 14, 1767, 16, 474, 584, 913, 4, 8, 655, 77, 301, 674, 19, 2, 917, 567, 50, 563, 1, 31, 0, 157, 9, 5492],
    [1858, 58, 17, 26, 2529, 28, 5, 19, 1628, 5, 12, 143, 105, 76, 1639, 16, 2, 342, 228, 372, 561, 2, 66, 0, 273, 3, 966],
    [1502, 482, 2418, 1244, 3245, 666, 1086, 22, 52, 26, 197, 2109, 1193, 10424, 1465, 651, 92, 1153, 3144, 2473, 161, 853, 22, 127, 10, 691, 517],
    [306, 0, 0, 0, 208, 1, 0, 0, 106, 0, 1, 0, 0, 0, 288, 1, 0, 1, 2, 0, 277, 0, 1, 0, 0, 0, 7],
    [330, 47, 9, 21, 1462, 28, 9, 85, 1017, 4, 31, 256, 29, 169, 155, 24, 0, 72, 603, 19, 92, 0, 47, 0, 159, 0, 768],
    [2969, 126, 139, 479, 4886, 146, 104, 29, 3835, 5, 195, 2477, 175, 49, 2114, 142, 1, 42, 1004, 517, 944, 173, 28, 12, 1730, 7, 1745],
    [2385, 688, 70, 11, 2134, 34, 2, 16, 1854, 0, 2, 59, 500, 99, 1372, 1046, 5, 22, 512, 12, 560, 4, 10, 0, 128, 1, 850],
    [1673, 137, 1370, 2164, 3384, 321, 6705, 141, 2065, 70, 578, 178, 69, 699, 1099, 81, 49, 99, 2232, 2831, 435, 214, 71, 24, 210, 62, 3427],
    [609, 542, 867, 819, 296, 308, 634, 101, 582, 26, 380, 1780, 1329, 4149, 1417, 940, 27, 3528, 1381, 1345, 2009, 711, 1070, 172, 244, 97, 777],
    [1537, 29, 25, 20, 2150, 19, 12, 563, 1367, 4, 17, 1004, 30, 26, 1317, 696, 0, 1323, 599, 412, 614, 1, 18, 0, 122, 1, 458],
    [4, 1, 0, 0, 0, 0, 0, 0, 7, 0, 0, 4, 0, 0, 1, 0, 0, 0, 1, 0, 862, 0, 1, 0, 0, 0, 8],
    [3800, 371, 488, 928, 5622, 170, 478, 112, 4061, 29, 426, 444, 666, 676, 2887, 272, 27, 806, 2626, 1316, 1060, 228, 93, 7, 723, 12, 3914],
    [1229, 90, 1113, 47, 2898, 56, 38, 2029, 1956, 9, 397, 626, 575, 283, 1069, 1093, 168, 40, 2047, 5100, 1056, 20, 323, 0, 253, 5, 17282],
    [2292, 90, 380, 26, 5506, 97, 33, 1626, 4257, 0, 11, 561, 83, 102, 1994, 51, 0, 1810, 1987, 1192, 950, 13, 204, 0, 672, 106, 3947],
    [575, 515, 637, 568, 672, 224, 466, 17, 619, 18, 79, 1403, 1046, 2158, 106, 555, 5, 1970, 2033, 1242, 11, 42, 7, 50, 27, 90, 153],
    [752, 1, 2, 5, 2260, 1, 2, 3, 1103, 0, 0, 11, 0, 2, 371, 2, 0, 7, 13, 3, 65, 17, 0, 0, 29, 0, 65],
    [1067, 50, 11, 71, 808, 20, 11, 316, 826, 0, 36, 136, 19, 269, 534, 23, 0, 167, 259, 23, 14, 0, 10, 0, 29, 6, 251],
    [77, 2, 99, 0, 220, 2, 2, 33, 225, 0, 0, 5, 3, 2, 49, 146, 1, 2, 2, 146, 37, 16, 6, 30, 27, 1, 200],
    [273, 81, 119, 70, 340, 28, 49, 29, 295, 6, 15, 159, 176, 161, 157, 179, 1, 135, 429, 75, 51, 5, 60, 12, 3, 19, 4160],
    [161, 6, 2, 3, 761, 0, 1, 16, 294, 0, 3, 76, 9, 5, 114, 5, 1, 2, 3, 6, 24, 3, 6, 0, 42, 181, 132],
    [3511, 3972, 5497, 3575, 2322, 2776, 2290, 2367, 1825, 806, 741, 2250, 3460, 1212, 1383, 4479, 293, 3348, 7334, 3185, 1175, 1015, 1833, 65, 276, 181, 0],])


def reference_logits():
    """Logits of the pretrained model: log of the bigram counts, smoothed by 1/2."""
    logits = np.log(COUNTS + 0.5)
    return logits - logits.max(axis=1, keepdims=True)


def softmax(logits):
    """Softmax of each row."""
    z = np.exp(logits - logits.max(axis=-1, keepdims=True))
    return z / z.sum(axis=-1, keepdims=True)


def sample(logits, n, rng=None):
    """Sample `n` words from the bigram model. Returns an integer array of shape
    (n, MAX_LENGTH + 1): the tokens of each word, ending with END, then padded with -1. A word
    is cut at MAX_LENGTH letters."""
    rng = np.random.default_rng(rng)
    cdf = softmax(logits).cumsum(axis=1)
    words = np.full((n, MAX_LENGTH + 1), -1)
    previous = np.full(n, START)
    alive = np.ones(n, dtype=bool)
    for t in range(MAX_LENGTH + 1):
        token = np.minimum((rng.random(n)[:, None] > cdf[previous]).sum(axis=1), END)
        if t == MAX_LENGTH:
            token[:] = END
        words[alive, t] = token[alive]
        alive &= token != END
        previous = np.where(alive, token, previous)
        if not alive.any():
            break
    return words


def encode(word):
    """Tokens of a string of letters, with the end token."""
    return np.array([ALPHABET.index(c) for c in word] + [END])


def decode(tokens):
    """String of the letters of a word given by its tokens."""
    return "".join(ALPHABET[t] for t in tokens if 0 <= t < END)


def transitions(tokens):
    """Matrix of shape (27, 27) counting the transitions (previous token, token) of a word:
    log pi(word) = sum of transitions(word) * log softmax(L)."""
    counts = np.zeros((27, 27))
    previous = START
    for t in tokens:
        if t < 0:
            break
        counts[previous, t] += 1
        if t == END:
            break
        previous = t
    return counts


def taste(word):
    """Score of a word (a string) for the annotators: soft letters (a, e, i, o, u, l, m, n)
    are liked, hard ones (j, k, q, v, x, z) disliked, runs of three vowels or three consonants
    are hard to pronounce, a pair of letters that comes back is a boring repetition, and the
    ideal length is 6. The learner never sees this function, only comparisons."""
    if not word:
        return -3.0
    soft = sum(c in "aeioulmn" for c in word) / len(word)
    hard = sum(c in "jkqvxz" for c in word)
    runs = len(re.findall(r"[^aeiou]{3,}|[aeiou]{3,}", word))
    pairs = [word[i:i + 2] for i in range(len(word) - 1)]
    repeats = len(pairs) - len(set(pairs))
    return 3 * soft - hard - 1.5 * runs - repeats - 0.3 * abs(len(word) - 6)


def compare(word_a, word_b, rng=None):
    """True if the annotators prefer word_a to word_b: Bradley-Terry model on taste."""
    rng = np.random.default_rng(rng)
    return bool(rng.random() < 1 / (1 + np.exp(taste(word_b) - taste(word_a))))


def preference_data(n, rng=None):
    """`n` comparisons of two words sampled from the reference model. Returns (chosen,
    rejected), two integer arrays of tokens of shape (n, MAX_LENGTH + 1)."""
    rng = np.random.default_rng(rng)
    a, b = sample(reference_logits(), n, rng), sample(reference_logits(), n, rng)
    first = np.array([compare(decode(x), decode(y), rng) for x, y in zip(a, b)])
    return np.where(first[:, None], a, b), np.where(first[:, None], b, a)


FEATURE_NAMES = list(ALPHABET) + ["length", "length^2/10"]


def word_features(word):
    """Features of a word (a string) for the reward model: the count of each letter, the
    length, and the squared length divided by 10."""
    features = np.zeros(len(FEATURE_NAMES))
    for c in word:
        features[ALPHABET.index(c)] += 1
    features[26], features[27] = len(word), len(word) ** 2 / 10
    return features

In [ ]:
%%writefile seqlearn/plotting.py
"""Shared plotting: course palette and mean curves with uncertainty bands."""

from math import lgamma

import matplotlib.pyplot as plt
import numpy as np
from cycler import cycler

# Categorical palette, to be used in this order (validated for color vision deficiencies on
# adjacent pairs). Beyond 4 series, prefer several subplots.
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
TEXT = "#52514e"
MUTED = "#a3a29c"
GRID = "#e6e5e1"


def set_style():
    plt.rcParams.update({
        "axes.prop_cycle": cycler(color=COLORS),
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": MUTED,
        "axes.labelcolor": TEXT,
        "axes.titlecolor": TEXT,
        "axes.grid": True,
        "axes.axisbelow": True,
        "grid.color": GRID,
        "grid.linewidth": 0.8,
        "xtick.color": TEXT,
        "ytick.color": TEXT,
        "lines.linewidth": 2,
        "lines.solid_capstyle": "round",
        "legend.frameon": False,
        "legend.labelcolor": TEXT,
        "font.size": 10,
        "figure.dpi": 110,
        "savefig.bbox": "tight",
    })


def plot_mean(ax, curves, label=None, band="se", color=None, end_label=False, label_dy=0, **kwargs):
    """Plot the mean of `curves`, of shape (n_runs, horizon), with an uncertainty band.

    band="se": mean +/- 2 standard errors; band="quantile": 10% and 90% quantiles.
    end_label=True writes `label` at the end of the curve, in addition to the legend,
    shifted vertically by `label_dy` points to avoid overlaps.
    """
    curves = np.asarray(curves, dtype=float)
    t = np.arange(1, curves.shape[1] + 1)
    mean = curves.mean(axis=0)
    if band == "se":
        half = 2 * curves.std(axis=0, ddof=1) / np.sqrt(len(curves))
        low, high = mean - half, mean + half
    elif band == "quantile":
        low, high = np.quantile(curves, [0.1, 0.9], axis=0)
    else:
        raise ValueError("band must be 'se' or 'quantile'.")
    (line,) = ax.plot(t, mean, label=label, color=color, **kwargs)
    ax.fill_between(t, low, high, color=line.get_color(), alpha=0.15, linewidth=0)
    if end_label and label:
        ax.annotate(label, xy=(t[-1], mean[-1]), xytext=(6, label_dy), textcoords="offset points",
                    va="center", color=TEXT, fontsize="small")
    return line


def beta_pdf(x, a, b):
    """Density of the Beta(a, b) distribution at the points `x` of [0, 1]."""
    x = np.clip(np.asarray(x, dtype=float), 1e-12, 1 - 1e-12)
    log_norm = lgamma(a + b) - lgamma(a) - lgamma(b)
    return np.exp(log_norm + (a - 1) * np.log(x) + (b - 1) * np.log(1 - x))

In [ ]:
%%writefile seqlearn/checks.py
"""Automatic tests for the labs. Each function raises AssertionError with an explicit message."""

import numpy as np

from .bandits import Bandit, run


class _FixedBandit(Bandit):
    """Deterministic bandit: arm a always pays means[a]."""

    def pull(self, arm):
        return float(self.means[arm])


class _DecayBandit(Bandit):
    """Arm 0 pays 1 at its first pull and 0 afterwards; arm 1 always pays 0.5."""

    def __init__(self):
        super().__init__([0.0, 0.5])
        self._first = True

    def pull(self, arm):
        if arm == 1:
            return 0.5
        reward, self._first = float(self._first), False
        return reward


def _arms(agent, env, horizon):
    arms, _ = run(env, agent, horizon)
    return arms.tolist()


def _ok(name):
    print(f"{name}: all tests passed.")


def check_pseudo_regret(pseudo_regret):
    gaps = np.array([0.0, 0.1, 0.3])
    arms = np.array([0, 1, 1, 2])
    expected = np.array([0.0, 0.1, 0.2, 0.5])
    out = pseudo_regret(arms, gaps)
    assert out is not None, "pseudo_regret returns nothing."
    assert np.allclose(out, expected), (
        f"For arms={arms.tolist()} and gaps={gaps.tolist()}, "
        f"expected {expected.tolist()}, got {np.asarray(out).tolist()}."
    )
    batch = np.stack([arms, arms[::-1]])
    out = pseudo_regret(batch, gaps)
    assert np.shape(out) == (2, 4), "For arms of shape (n_runs, T), the output must have the same shape."
    assert np.allclose(out, [expected, [0.3, 0.4, 0.5, 0.5]]), (
        "The cumulative sum must run along the last axis (time)."
    )
    _ok("pseudo_regret")


def check_greedy(GreedyAgent):
    got = _arms(GreedyAgent(3), _FixedBandit([0.2, 0.8, 0.5]), 6)
    assert got == [0, 1, 2, 1, 1, 1], (
        f"Noiseless means (0.2, 0.8, 0.5): expected [0, 1, 2, 1, 1, 1], got {got}. "
        "Each arm must be pulled once, in order, before taking the argmax."
    )
    got = _arms(GreedyAgent(2), _FixedBandit([0.5, 0.5]), 4)
    assert got == [0, 1, 0, 0], (
        f"Equal means: expected [0, 1, 0, 0], got {got}. Ties must go to the smallest index."
    )
    agent = GreedyAgent(2)
    _arms(agent, _FixedBandit([0.9, 0.1]), 5)
    got = _arms(agent, _FixedBandit([0.1, 0.9]), 4)
    assert got == [0, 1, 1, 1], "reset() must bring the agent back to its initial state."
    _ok("GreedyAgent")


def check_epsilon_greedy(EpsilonGreedyAgent):
    got = _arms(EpsilonGreedyAgent(3, 0.0), _FixedBandit([0.2, 0.8, 0.5]), 6)
    assert got == [0, 1, 2, 1, 1, 1], f"With epsilon = 0, the agent must behave like greedy: got {got}."
    horizon = 4002
    arms = np.array(_arms(EpsilonGreedyAgent(3, 1.0, rng=0), _FixedBandit([0.2, 0.8, 0.5]), horizon))
    freqs = np.bincount(arms[3:], minlength=3) / (horizon - 3)
    assert np.allclose(freqs, 1 / 3, atol=0.04), (
        f"With epsilon = 1, arms must be uniform: frequencies {freqs.round(3)}."
    )
    arms = np.array(_arms(EpsilonGreedyAgent(2, 0.3, rng=0), _FixedBandit([0.0, 1.0]), horizon))
    freq = np.mean(arms[2:] == 0)
    assert abs(freq - 0.15) < 0.03, (
        "With epsilon = 0.3 and two arms, the bad arm must be played about 15% of the time "
        f"(uniform exploration, hence epsilon / 2): got {100 * freq:.1f}%."
    )
    _ok("EpsilonGreedyAgent")


def check_etc(ETCAgent):
    got = _arms(ETCAgent(3, 2), _FixedBandit([0.3, 0.7, 0.5]), 9)
    assert got == [0, 1, 2, 0, 1, 2, 1, 1, 1], (
        f"ETC(m=2) on (0.3, 0.7, 0.5): expected [0, 1, 2, 0, 1, 2, 1, 1, 1], got {got}. "
        "During exploration, arms are played in turn."
    )
    got = _arms(ETCAgent(2, 3), _FixedBandit([0.5, 0.5]), 8)
    assert got == [0, 1, 0, 1, 0, 1, 0, 0], (
        f"Equal means: expected [0, 1, 0, 1, 0, 1, 0, 0], got {got}. Ties must go to the smallest index."
    )
    got = _arms(ETCAgent(2, 1), _DecayBandit(), 6)
    assert got == [0, 1, 0, 0, 0, 0], (
        f"Expected [0, 1, 0, 0, 0, 0], got {got}. After exploration, ETC plays the same arm "
        "until the end, even if the empirical means change."
    )
    _ok("ETCAgent")


# ---------------------------------------------------------------------------
# Lab 2

def check_ucb(UCBAgent):
    agent = UCBAgent(2, alpha=2.0)
    for arm, reward in [(0, 1.0), (1, 0.0), (0, 0.0)]:
        agent.update(arm, reward)
    out = agent.index()
    assert out is not None, "index returns nothing."
    assert np.shape(out) == (2,), f"index must return an array of length n_arms: got shape {np.shape(out)}."
    expected = np.array([0.5, 0.0]) + np.sqrt(2.0 * np.log(4) / np.array([2, 1]))
    shifted = np.array([0.5, 0.0]) + np.sqrt(2.0 * np.log(3) / np.array([2, 1]))
    assert not np.allclose(out, shifted), (
        "After 3 rounds, the index is the one of round t = 4: the logarithm is log(self.t + 1), "
        "not log(self.t)."
    )
    assert not np.allclose(out, [0.5, 0.0]), "The index is the empirical mean alone: the bonus is missing."
    assert np.allclose(out, expected), (
        f"After the pulls (arm 0, reward 1), (arm 1, reward 0), (arm 0, reward 0), with alpha = 2, "
        f"expected {expected.round(3).tolist()}, got {np.asarray(out).round(3).tolist()}."
    )
    agent = UCBAgent(2, alpha=0.5)
    for arm, reward in [(0, 1.0), (1, 0.0), (0, 0.0)]:
        agent.update(arm, reward)
    expected = np.array([0.5, 0.0]) + np.sqrt(0.5 * np.log(4) / np.array([2, 1]))
    assert np.allclose(agent.index(), expected), (
        f"With alpha = 0.5, expected {expected.round(3).tolist()}, "
        f"got {np.asarray(agent.index()).round(3).tolist()}. The bonus must depend on self.alpha."
    )
    got = _arms(UCBAgent(3, alpha=2.0), _FixedBandit([0.2, 0.8, 0.5]), 3)
    assert got == [0, 1, 2], f"During the first K rounds, each arm must be pulled once, in order: got {got}."
    got = _arms(UCBAgent(3, alpha=2.0), _FixedBandit([0.2, 0.8, 0.5]), 12)
    expected = [0, 1, 2, 1, 2, 1, 0, 1, 2, 1, 1, 2]
    assert got == expected, (
        f"Noiseless means (0.2, 0.8, 0.5), alpha = 2: expected {expected}, got {got}. "
        "After the first K rounds, play the argmax of the index."
    )
    got = _arms(UCBAgent(2, alpha=0.5), _FixedBandit([0.9, 0.1]), 14)
    expected = [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
    assert got == expected, f"Noiseless means (0.9, 0.1), alpha = 0.5: expected {expected}, got {got}."
    agent = UCBAgent(2, alpha=2.0)
    _arms(agent, _FixedBandit([0.9, 0.1]), 5)
    got = _arms(agent, _FixedBandit([0.1, 0.9]), 4)
    assert got == [0, 1, 1, 1], "reset() must bring the agent back to its initial state."
    _ok("UCBAgent")


# ---------------------------------------------------------------------------
# Lab 3

def _feed(agent, history):
    for arm, reward in history:
        agent.update(arm, reward)


def check_thompson(ThompsonAgent):
    agent = ThompsonAgent(2, rng=0)
    _feed(agent, [(0, 1.0), (0, 1.0), (0, 0.0), (1, 0.0)])
    out = agent.posterior()
    assert out is not None and len(out) == 2, "posterior must return the pair (alpha, beta)."
    alpha, beta = np.asarray(out[0], dtype=float), np.asarray(out[1], dtype=float)
    assert alpha.shape == (2,) and beta.shape == (2,), "alpha and beta must be arrays of length n_arms."
    assert np.allclose(alpha, [3, 1]) and np.allclose(beta, [2, 2]), (
        "After 2 successes and 1 failure on arm 0, and 1 failure on arm 1, with uniform priors, "
        f"expected alpha = [3, 1] and beta = [2, 2], got alpha = {alpha.tolist()} and beta = {beta.tolist()}."
    )
    agent = ThompsonAgent(3, rng=0)
    draws = np.array([agent.act() for _ in range(3000)])
    assert np.all(np.isin(draws, [0, 1, 2])), "act() must return an arm index."
    freqs = np.bincount(draws, minlength=3) / len(draws)
    assert np.allclose(freqs, 1 / 3, atol=0.04), (
        f"Before any observation, the three arms must be played with the same probability: frequencies {freqs.round(3)}. "
        "Thompson sampling has no initialization rounds."
    )
    agent = ThompsonAgent(2, rng=0)
    _feed(agent, [(0, 1.0)] * 60 + [(1, 0.0)] * 60)
    draws = [agent.act() for _ in range(200)]
    assert all(arm == 0 for arm in draws), "After 60 successes on arm 0 and 60 failures on arm 1, arm 0 must always be played."
    agent = ThompsonAgent(2, rng=0)
    _feed(agent, [(0, 1.0), (1, 0.0)])
    freq = np.mean([agent.act() == 0 for _ in range(6000)])
    assert abs(freq - 5 / 6) < 0.02, (
        "After one success on arm 0 and one failure on arm 1, arm 0 must be played with probability 5/6 "
        f"(Tutorial 3, exercise 1): got {freq:.3f}."
    )
    first, second = ThompsonAgent(2, rng=7), ThompsonAgent(2, rng=7)
    assert [first.act() for _ in range(50)] == [second.act() for _ in range(50)], (
        "Two agents with the same seed must make the same choices: use self.rng for all random draws."
    )
    _ok("ThompsonAgent")


def _ridge_data():
    rng = np.random.default_rng(0)
    xs = rng.standard_normal((6, 3))
    rs = rng.random(6)
    lam = 2.0
    V = lam * np.eye(3) + xs.T @ xs
    b = xs.T @ rs
    return xs, rs, lam, V, b


def check_ridge(Ridge):
    xs, rs, lam, V, b = _ridge_data()
    ridge = Ridge(3, lam)
    assert np.allclose(ridge.V_inv, np.eye(3) / lam), "Before any observation, V_inv must be the identity divided by lam."
    assert np.allclose(ridge.theta_hat, 0), "Before any observation, theta_hat must be zero."
    for x, r in zip(xs, rs):
        ridge.update(x, r)
    assert np.allclose(ridge.V_inv, np.linalg.inv(V)), (
        "After 6 updates, V_inv differs from the inverse of lam * I + sum of x x^T: check the Sherman-Morrison update."
    )
    assert np.allclose(ridge.b, b), "After 6 updates, b must be the sum of the r x."
    assert np.allclose(ridge.theta_hat, np.linalg.solve(V, b)), "theta_hat must be V_inv @ b, recomputed after each update."
    X = np.array([[1.0, 0.0, 0.0], [0.0, 2.0, 0.0], [1.0, 1.0, 1.0], [0.5, -1.0, 2.0]])
    out = ridge.width(X)
    assert out is not None, "width returns nothing."
    expected = np.sqrt(np.sum(X @ np.linalg.inv(V) * X, axis=1))
    assert np.shape(out) == (4,), f"For features of shape (4, 3), width must return an array of length 4: got shape {np.shape(out)}."
    assert np.allclose(out, expected), (
        f"width must return sqrt(x^T V_inv x) for each row x: expected {expected.round(3).tolist()}, got {np.asarray(out).round(3).tolist()}."
    )
    _ok("Ridge")


_LIN_HISTORY = [([1.0, 0.0], 1.0), ([1.0, 0.0], 1.0), ([1.0, 0.0], 0.0), ([1.0, 0.0], 0.0), ([0.0, 1.0], 0.0)]
_LIN_FEATURES = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])


def _feed_linear(agent):
    for x, reward in _LIN_HISTORY:
        agent.update(np.array(x), 0, reward)


def check_linucb(LinUCBAgent):
    agent = LinUCBAgent(2, beta=1.0, lam=1.0)
    out = agent.index(np.array([[1.0, 0.0], [2.0, 0.0]]))
    assert out is not None, "index returns nothing."
    assert np.allclose(out, [1.0, 2.0]), (
        f"Before any observation, with beta = 1 and lam = 1, the index of x is its norm: expected [1, 2], got {np.asarray(out).round(3).tolist()}."
    )
    _feed_linear(agent)
    out = np.asarray(agent.index(_LIN_FEATURES))
    expected = np.array([0.4 + np.sqrt(0.2), np.sqrt(0.5), 0.4 + np.sqrt(0.7)])
    assert np.allclose(out, expected), (
        "Example of Tutorial 3 (exercise 3): V = diag(5, 2), theta_hat = (0.4, 0), beta = 1. "
        f"Expected indices {expected.round(3).tolist()}, got {out.round(3).tolist()}."
    )
    assert agent.act(_LIN_FEATURES) == 2, "On the example of Tutorial 3, LinUCB must play the third arm."
    assert isinstance(agent.act(_LIN_FEATURES), (int, np.integer)), "act() must return an integer."
    agent = LinUCBAgent(2, beta=0.0, lam=1.0)
    _feed_linear(agent)
    assert agent.act(_LIN_FEATURES) == 0, "With beta = 0, the agent plays the argmax of <theta_hat, x>, ties going to the smallest index."
    agent.reset()
    assert np.allclose(agent.index(np.array([[1.0, 0.0]])), 0.0), "reset() must forget the observations."
    _ok("LinUCBAgent")


def check_lints(LinTSAgent):
    agent = LinTSAgent(2, v=0.5, lam=1.0, rng=0)
    _feed_linear(agent)
    samples = np.array([agent.sample() for _ in range(6000)])
    assert samples.shape == (6000, 2), "sample() must return a vector of length dim."
    mean, cov = samples.mean(axis=0), np.cov(samples.T)
    assert np.allclose(mean, [0.4, 0.0], atol=0.02), (
        f"The samples must be centered at theta_hat = (0.4, 0): empirical mean {mean.round(3).tolist()}."
    )
    expected = 0.25 * np.diag([0.2, 0.5])
    assert np.allclose(cov, expected, atol=0.01), (
        "The covariance of the samples must be v^2 V_inv = diag(0.05, 0.125): "
        f"empirical covariance {cov.round(3).tolist()}."
    )
    agent = LinTSAgent(2, v=0.0, lam=1.0, rng=0)
    _feed_linear(agent)
    assert all(agent.act(_LIN_FEATURES[:2]) == 0 for _ in range(20)), "With v = 0, the agent plays the argmax of <theta_hat, x>."
    first, second = LinTSAgent(2, v=1.0, lam=1.0, rng=3), LinTSAgent(2, v=1.0, lam=1.0, rng=3)
    assert [first.act(_LIN_FEATURES) for _ in range(30)] == [second.act(_LIN_FEATURES) for _ in range(30)], (
        "Two agents with the same seed must make the same choices: use self.rng for all random draws."
    )
    _ok("LinTSAgent")


# ---------------------------------------------------------------------------
# Lab 5

def check_hedge(hedge_probs):
    out = hedge_probs(np.zeros(3), 1.0)
    assert out is not None, "hedge_probs returns nothing."
    assert np.allclose(out, 1 / 3), f"With equal cumulative gains, the three probabilities must be 1/3: got {np.asarray(out).round(3).tolist()}."
    gains = np.array([2.0, 1.0, 0.0])
    out = np.asarray(hedge_probs(gains, np.log(2)))
    assert np.allclose(out, [4 / 7, 2 / 7, 1 / 7]), (
        "Tutorial 5, exercise 1: cumulative gains (2, 1, 0) and eta = log 2 give the weights (4, 2, 1), "
        f"so the probabilities (4/7, 2/7, 1/7). Got {out.round(3).tolist()}."
    )
    out = np.asarray(hedge_probs(np.array([5000.0, 4999.0]), 1.0))
    assert np.all(np.isfinite(out)), (
        "With large cumulative gains (5000, 4999), exp overflows: subtract the maximum of eta * gains before "
        "taking the exponential, which does not change the probabilities."
    )
    assert np.allclose(out, [np.e / (1 + np.e), 1 / (1 + np.e)]), f"Gains (5000, 4999), eta = 1: got {out.round(3).tolist()}."
    out = np.asarray(hedge_probs(gains, 0.0))
    assert np.allclose(out, 1 / 3), "With eta = 0, Hedge plays uniformly."
    _ok("hedge_probs")


def check_exp3(EXP3Agent):
    agent = EXP3Agent(3, eta=0.5, rng=0)
    assert np.allclose(agent.loss_hat, 0), "reset() must set loss_hat to zero."
    assert np.allclose(agent.probs(), 1 / 3), "Before any observation, EXP3 plays uniformly."
    agent.p = np.array([0.5, 0.25, 0.25])
    agent.update(1, 0.0)
    assert np.allclose(agent.loss_hat, [0, 4, 0]), (
        "Arm 1, played with probability 0.25, gives reward 0, that is loss 1: its estimated loss must increase by "
        f"1 / 0.25 = 4, and the other estimates must not change. Got loss_hat = {np.asarray(agent.loss_hat).round(3).tolist()}."
    )
    agent.p = np.array([0.5, 0.25, 0.25])
    agent.update(0, 1.0)
    assert np.allclose(agent.loss_hat, [0, 4, 0]), "A reward of 1 is a loss of 0: the estimates must not change."
    assert agent.counts[0] == 1 and agent.t == 2, "update must call super().update(arm, reward), which counts the rounds."
    expected = np.exp(-0.5 * np.array([0, 4, 0]))
    expected /= expected.sum()
    assert np.allclose(agent.probs(), expected), (
        f"probs() must be proportional to exp(-eta * loss_hat): expected {expected.round(3).tolist()}, "
        f"got {np.asarray(agent.probs()).round(3).tolist()}."
    )
    agent = EXP3Agent(2, eta=1.0, rng=0)
    agent.loss_hat = np.array([0.0, np.log(3)])
    draws = np.array([agent.act() for _ in range(6000)])
    assert np.allclose(agent.p, [0.75, 0.25]), "act() must store the probabilities of the round in self.p."
    assert abs(np.mean(draws == 0) - 0.75) < 0.02, (
        f"With probabilities (0.75, 0.25), arm 0 must be played 75% of the time: got {np.mean(draws == 0):.3f}."
    )
    agent = EXP3Agent(2, eta=1.0, rng=0)
    agent.loss_hat = np.array([0.0, 5000.0])
    assert np.all(np.isfinite(agent.probs())), "probs() must not overflow: subtract the minimum of loss_hat first."
    first, second = EXP3Agent(3, 0.1, rng=4), EXP3Agent(3, 0.1, rng=4)
    assert [first.act() for _ in range(40)] == [second.act() for _ in range(40)], (
        "Two agents with the same seed must make the same choices: use self.rng for all random draws."
    )
    _ok("EXP3Agent")


def check_ips(ips):
    logs = {"profiles": np.zeros((4, 2)), "items": np.array([0, 1, 0, 1]),
            "clicks": np.array([1.0, 1.0, 0.0, 1.0]), "propensities": np.array([0.5, 0.25, 0.5, 0.25])}

    def always_zero(profiles):
        return np.tile([1.0, 0.0], (len(profiles), 1))

    def half(profiles):
        return np.full((len(profiles), 2), 0.5)

    out = ips(logs, always_zero)
    assert out is not None, "ips returns nothing."
    assert np.isclose(out, 0.5), (
        "Policy that always plays item 0: the weights are (2, 0, 2, 0) and the clicks (1, 1, 0, 1), so the estimate "
        f"is (2 + 0 + 0 + 0) / 4 = 0.5. Got {float(out):.3f}."
    )
    assert np.isclose(ips(logs, half), (1 + 2 + 0 + 2) / 4), "Policy (0.5, 0.5): the weights are (1, 2, 1, 2)."
    assert np.isclose(ips(logs, half, clip=1.5), (1 + 1.5 + 0 + 1.5) / 4), (
        "With clip = 1.5, the weights (1, 2, 1, 2) become (1, 1.5, 1, 1.5)."
    )
    _ok("ips")


# ---------------------------------------------------------------------------
# Lab 6

class _CountingBandit(Bandit):
    """Deterministic bandit that counts the pulls of each arm: arm a always pays means[a]."""

    def __init__(self, means):
        super().__init__(means)
        self.pulls = np.zeros(self.n_arms, dtype=int)

    def pull(self, arm):
        self.pulls[arm] += 1
        return float(self.means[arm])


def check_uniform_allocation(uniform_allocation):
    env = _CountingBandit([0.2, 0.9, 0.5, 0.4])
    out = uniform_allocation(env, 40)
    assert out is not None, "uniform_allocation returns nothing."
    assert out == 1, f"Noiseless means (0.2, 0.9, 0.5, 0.4): the recommended arm must be 1, got {out}."
    assert env.pulls.tolist() == [10, 10, 10, 10], f"A budget of 40 on 4 arms gives 10 pulls each: got {env.pulls.tolist()}."
    env = _CountingBandit([0.2, 0.9, 0.5])
    uniform_allocation(env, 10)
    assert env.pulls.sum() == 10 and env.pulls.max() - env.pulls.min() <= 1, (
        f"With a budget of 10 on 3 arms, the pulls must be as balanced as possible and sum to 10: got {env.pulls.tolist()}."
    )
    _ok("uniform_allocation")


def check_sequential_halving(sequential_halving):
    env = _CountingBandit([0.3, 0.1, 0.8, 0.6])
    out = sequential_halving(env, 16)
    assert out is not None, "sequential_halving returns nothing."
    assert out == 2, f"Noiseless means (0.3, 0.1, 0.8, 0.6): the recommended arm must be 2, got {out}."
    assert env.pulls.tolist() == [2, 2, 6, 6], (
        "Budget 16, K = 4, so 2 phases of 8 pulls: 2 pulls of each arm, then 4 pulls of arms 2 and 3. "
        f"Expected the pull counts [2, 2, 6, 6], got {env.pulls.tolist()}."
    )
    env = _CountingBandit([0.5, 0.4, 0.3, 0.2, 0.1, 0.0, 0.6, 0.45])
    out = sequential_halving(env, 240)
    assert out == 6, f"Noiseless means with the best arm at index 6: got {out}."
    assert env.pulls.sum() <= 240, f"The budget is 240 pulls, but {env.pulls.sum()} were used."
    assert env.pulls.tolist() == [70, 30, 10, 10, 10, 10, 70, 30], (
        "Budget 240, K = 8, so 3 phases of 80 pulls: 10 pulls of each of the 8 arms, then 20 of each of the 4 best "
        f"(arms 6, 0, 7, 1), then 40 of each of the 2 best (arms 6 and 0). Got {env.pulls.tolist()}."
    )
    _ok("sequential_halving")


def _sigmoid(z):
    return 1 / (1 + np.exp(-z))


def check_bradley_terry(bt_log_likelihood, bt_gradient):
    wins = np.array([[0.0, 6.0], [2.0, 0.0]])
    out = bt_log_likelihood(np.zeros(2), wins)
    assert out is not None, "bt_log_likelihood returns nothing."
    assert np.isclose(out, 8 * np.log(0.5)), f"With equal scores, each of the 8 duels has probability 1/2: expected 8 log(1/2), got {float(out):.4f}."
    s = np.array([np.log(3), 0.0])
    assert np.isclose(bt_log_likelihood(s, wins), 6 * np.log(0.75) + 2 * np.log(0.25)), (
        "Scores (log 3, 0): model 0 wins with probability 3/4, and the log-likelihood is 6 log(3/4) + 2 log(1/4)."
    )
    grad = bt_gradient(s, wins)
    assert grad is not None, "bt_gradient returns nothing."
    assert np.allclose(grad, 0), (
        "Tutorial 6, exercise 2: with 6 wins out of 8, the maximum is at s_0 - s_1 = log 3, where the gradient vanishes. "
        f"Got {np.asarray(grad).round(4).tolist()}."
    )
    rng = np.random.default_rng(0)
    wins = rng.integers(0, 5, (4, 4)).astype(float)
    np.fill_diagonal(wins, 0)
    s = rng.normal(size=4)
    eps = 1e-6
    numeric = np.array([(bt_log_likelihood(s + eps * e, wins) - bt_log_likelihood(s - eps * e, wins)) / (2 * eps) for e in np.eye(4)])
    assert np.allclose(bt_gradient(s, wins), numeric, atol=1e-5), (
        f"bt_gradient differs from the numerical derivative of bt_log_likelihood: expected {numeric.round(3).tolist()}, "
        f"got {np.asarray(bt_gradient(s, wins)).round(3).tolist()}."
    )
    _ok("bt_log_likelihood and bt_gradient")


def check_elo(elo_update):
    ratings = np.zeros(3)
    out = elo_update(ratings, 0, 1, 1.0, 1.0)
    assert out is not None, "elo_update must return the new ratings."
    assert np.allclose(out, [0.5, -0.5, 0.0]), f"Equal ratings, model 0 wins, k = 1: expected [0.5, -0.5, 0], got {np.asarray(out).round(4).tolist()}."
    assert np.allclose(ratings, 0), "elo_update must not modify its argument: work on a copy."
    out = elo_update(np.array([0.5, -0.5, 0.0]), 1, 2, 1.0, 1.0)
    assert np.allclose(out, [0.5, -0.5 + (1 - _sigmoid(-0.5)), -(1 - _sigmoid(-0.5))]), (
        f"Tutorial 6, exercise 2: after B beats C, expected ratings {np.round([0.5, -0.5 + (1 - _sigmoid(-0.5)), -(1 - _sigmoid(-0.5))], 4).tolist()}, "
        f"got {np.asarray(out).round(4).tolist()}."
    )
    out = elo_update(np.array([1.0, 0.0]), 0, 1, 0.0, 0.5)
    assert np.allclose(out, [1.0 - 0.5 * _sigmoid(1.0), 0.5 * _sigmoid(1.0)]), "A loss of model 0 must decrease its rating by k sigmoid(r_0 - r_1)."
    _ok("elo_update")


# ---------------------------------------------------------------------------
# Lab 7

def _three_states():
    """MDP of Tutorial 7, exercise 2: states 0, 1, 2, actions 0 (advance) and 1 (stay or go back)."""
    P = np.zeros((3, 2, 3))
    R = np.zeros((3, 2))
    P[0, 0, 1] = P[0, 1, 0] = 1.0
    R[0, 1] = 1.0
    P[1, 0, 2] = P[1, 1, 0] = 1.0
    P[2, :, 0] = 1.0
    R[2, :] = 4.0
    return P, R


def check_policy_evaluation(policy_evaluation):
    P, R = _three_states()
    out = policy_evaluation(P, R, np.array([0, 0, 0]), 0.5)
    assert out is not None, "policy_evaluation returns nothing."
    v1 = 4 * 0.25 / (1 - 0.125)
    expected = np.array([v1, 2 * v1, 4 + 0.5 * v1])
    assert np.allclose(out, expected), (
        "MDP of Tutorial 7, exercise 2: for the policy that always advances, with gamma = 1/2, V(0) = 4 gamma^2 / (1 - gamma^3) = 8/7, "
        f"V(1) = 16/7, V(2) = 32/7. Expected {expected.round(4).tolist()}, got {np.asarray(out).round(4).tolist()}."
    )
    out = policy_evaluation(P, R, np.array([1, 1, 0]), 0.5)
    assert np.isclose(out[0], 2.0), f"The policy that stays in state 0 has V(0) = 1 / (1 - gamma) = 2: got {float(out[0]):.4f}."
    _ok("policy_evaluation")


def check_value_iteration(value_iteration):
    P, R = _three_states()
    out = value_iteration(P, R, 0.5, n_iter=2)
    assert out is not None and len(out) == 2, "value_iteration must return the pair (V, Q)."
    V, Q = out
    assert np.allclose(V, [1.5, 2.0, 4.5]), (
        f"MDP of Tutorial 7, exercise 2: two iterations from V = 0 with gamma = 1/2 give V = (1.5, 2, 4.5). Got {np.asarray(V).round(4).tolist()}."
    )
    assert np.shape(Q) == (3, 2), f"Q must have shape (n_states, n_actions): got {np.shape(Q)}."
    assert np.allclose(Q, R + 0.5 * P @ np.array([1.0, 0.0, 4.0])), (
        "Q must be the Q-values of the last iteration, R + gamma P V computed with the V of the previous iteration."
    )
    V, Q = value_iteration(P, R, 0.9, n_iter=500)
    v1 = 4 * 0.81 / (1 - 0.729)
    assert np.allclose(V, [v1, (v1 - 0) / 0.9, 4 + 0.9 * v1], atol=1e-6), (
        "With gamma = 0.9, the optimal policy always advances: V(0) = 4 gamma^2 / (1 - gamma^3). "
        f"Got V = {np.asarray(V).round(3).tolist()}."
    )
    _ok("value_iteration")


def check_qlearning(QLearningAgent):
    agent = QLearningAgent(3, 2, alpha=0.5, gamma=0.5, epsilon=0.0, rng=0)
    assert np.allclose(agent.Q, 0) and np.shape(agent.Q) == (3, 2), "Q must start at zero, with shape (n_states, n_actions)."
    for s, a, r, s_next in [(0, 0, 0.0, 1), (1, 0, 0.0, 2), (2, 0, 4.0, 0), (0, 0, 0.0, 1), (1, 0, 0.0, 2)]:
        agent.update(s, a, r, s_next, False)
    expected = np.zeros((3, 2))
    expected[2, 0], expected[1, 0] = 2.0, 0.5
    assert np.allclose(agent.Q, expected), (
        "Tutorial 7, exercise 2: after the trajectory, Q(2, advance) = 2, Q(1, advance) = 0.5 and the other values are 0. "
        f"Got {np.asarray(agent.Q).round(3).tolist()}."
    )
    agent.update(2, 1, 4.0, 0, True)
    assert np.isclose(agent.Q[2, 1], 2.0), "At the end of an episode (done = True), the target is the reward alone."
    assert agent.act(1) == 0, "With epsilon = 0, act plays the greedy action."
    agent = QLearningAgent(3, 2, alpha=0.5, gamma=0.5, epsilon=1.0, rng=0)
    draws = [agent.act(0) for _ in range(2000)]
    assert abs(np.mean(draws) - 0.5) < 0.05, "With epsilon = 1, act plays uniformly at random."
    first, second = QLearningAgent(3, 2, epsilon=0.5, rng=3), QLearningAgent(3, 2, epsilon=0.5, rng=3)
    assert [first.act(0) for _ in range(50)] == [second.act(0) for _ in range(50)], (
        "Two agents with the same seed must make the same choices: use self.rng for all random draws."
    )
    _ok("QLearningAgent")


# ---------------------------------------------------------------------------
# Lab 0

def check_gaps(gaps):
    out = gaps(np.array([0.5, 0.6, 0.4]))
    assert out is not None, "gaps returns nothing."
    assert np.allclose(out, [0.1, 0.0, 0.2]), (
        f"For means (0.5, 0.6, 0.4), expected [0.1, 0.0, 0.2], got {np.asarray(out).tolist()}."
    )
    _ok("gaps")


def check_counts(counts):
    out = counts(np.array([0, 2, 2, 0, 2]), 4)
    assert out is not None, "counts returns nothing."
    assert np.shape(out) == (4,), (
        f"With n_arms = 4, the output must have length 4, even if an arm is never played: got shape {np.shape(out)}."
    )
    assert np.array_equal(out, [2, 0, 3, 0]), (
        f"For arms [0, 2, 2, 0, 2] and n_arms = 4, expected [2, 0, 3, 0], got {np.asarray(out).tolist()}."
    )
    _ok("counts")


def check_running_mean(running_mean):
    x = np.array([1.0, 0.0, 1.0, 1.0])
    expected = np.array([1.0, 0.5, 2 / 3, 0.75])
    out = running_mean(x)
    assert out is not None, "running_mean returns nothing."
    assert np.allclose(out, expected), (
        f"For x = {x.tolist()}, expected {expected.round(3).tolist()}, got {np.asarray(out).round(3).tolist()}."
    )
    batch = np.stack([x, 1 - x])
    out = running_mean(batch)
    assert np.shape(out) == (2, 4), "For x of shape (n_runs, T), the output must have the same shape."
    assert np.allclose(out, [expected, 1 - expected]), "The running mean must be computed along the last axis (time)."
    _ok("running_mean")


def check_bernoulli(bernoulli):
    out = bernoulli(0.3, 20000, np.random.default_rng(0))
    assert out is not None, "bernoulli returns nothing."
    out = np.asarray(out)
    assert out.shape == (20000,), f"With size = 20000, the output must have shape (20000,): got {out.shape}."
    assert np.all((out == 0) | (out == 1)), "The values must be 0 or 1."
    assert abs(out.mean() - 0.3) < 0.02, f"With p = 0.3, the mean of the draws must be close to 0.3: got {out.mean():.3f}."
    again = np.asarray(bernoulli(0.3, 20000, np.random.default_rng(0)))
    assert np.array_equal(out, again), "Two calls with generators of the same seed must return the same draws: use `rng` for all random draws."
    out = np.asarray(bernoulli(0.5, (3, 7), np.random.default_rng(1)))
    assert out.shape == (3, 7), f"With size = (3, 7), the output must have shape (3, 7): got {out.shape}."
    _ok("bernoulli")


def check_running_mean_class(RunningMean):
    m = RunningMean()
    assert m.n == 0, "A new RunningMean must have n = 0."
    values = []
    for x in [1.0, 0.0, 1.0, 1.0]:
        m.update(x)
        values.append(m.value)
    assert m.n == 4, f"After 4 updates, n must be 4: got {m.n}."
    assert np.allclose(values, [1.0, 0.5, 2 / 3, 0.75]), (
        f"After the updates 1, 0, 1, 1, the successive values must be [1, 0.5, 0.667, 0.75]: got {np.round(values, 3).tolist()}."
    )
    stored = [v for v in vars(m).values() if isinstance(v, (list, np.ndarray)) and np.size(v) > 1]
    assert not stored, "The past observations must not be stored: keep only n and value."
    _ok("RunningMean")


def check_round_robin(RoundRobinAgent):
    got = _arms(RoundRobinAgent(3), _FixedBandit([0.2, 0.8, 0.5]), 7)
    assert got == [0, 1, 2, 0, 1, 2, 0], f"With 3 arms, expected [0, 1, 2, 0, 1, 2, 0], got {got}."
    _ok("RoundRobinAgent")


# ---------------------------------------------------------------------------
# Lab 8

def check_linear_q(LinearQAgent):
    from .blocks import Blocks, placement_features

    game = Blocks(width=3, height=4, pieces=[["#"]], rng=0)
    board, piece = game.reset()
    agent = LinearQAgent(game, n_features=8, alpha=0.5, gamma=0.5, epsilon=0.0, rng=0)
    assert np.shape(agent.w) == (8,) and np.allclose(agent.w, 0), "w must start at zero."
    agent.w = np.array([0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0])  # rewards the height of column 0
    actions, phi, lost, q = agent.q_values(board, piece)
    assert len(q) == len(actions) == 3, "q_values must return one Q-value per legal placement."
    assert np.allclose(q, [0.25, 0.0, 0.0]), (
        f"With w = e_1, Q(s, a) is the height of column 0 after the placement divided by 4: expected [0.25, 0, 0], got {np.round(q, 3).tolist()}."
    )
    action, phi0 = agent.act(board, piece)
    assert action == actions[0] and np.allclose(phi0, phi[0]), "With epsilon = 0, act returns the greedy placement and its features."
    full = np.zeros((4, 3), dtype=bool)
    full[:, 1] = True
    _, _, lost, q = agent.q_values(full, piece)
    assert lost[1] and q[1] == -1.0, "A placement that ends the game has Q-value -1, its reward."
    agent.w = -10.0 * np.eye(8)[0]  # every placement that goes on is worth -10
    action, phi1 = agent.act(full, piece)
    assert action == actions[1] and phi1 is None, "act returns None as features for a placement that ends the game."
    phi = placement_features(game, board, piece)[1][0]
    agent.w = np.eye(8)[0].copy()
    agent.update(phi, 1.0, board, piece, False)
    target = 1.0 + 0.5 * 1.0
    expected = np.eye(8)[0] + 0.5 * (target - 1.0) * phi
    assert np.allclose(agent.w, expected), (
        "update: w + alpha (r + gamma max_a' Q(s', a') - w.phi) phi, with w = e_0, r = 1, gamma = 1/2, so the target is 1 + 1/2 = 1.5. "
        f"Expected {expected.round(3).tolist()}, got {np.round(agent.w, 3).tolist()}."
    )
    agent.w = np.zeros(8)
    agent.update(phi, 1.0, board, piece, True)
    assert np.allclose(agent.w, 0.5 * phi), "At the end of the game (done = True), the target is the reward alone."
    agent = LinearQAgent(game, n_features=8, epsilon=1.0, rng=0)
    draws = [actions.index(agent.act(board, piece)[0]) for _ in range(600)]
    assert np.bincount(draws, minlength=3).min() > 150, "With epsilon = 1, act plays a uniformly random placement."
    _ok("LinearQAgent")


def check_policy(policy, grad_log_policy):
    W = np.array([[0.0, 1.0, -1.0, 2.0], [0.5, 0.0, 1.0, -1.0]])
    s = np.array([0.1, -0.2, 0.3, 0.05])
    z = W @ s
    expected = np.exp(z) / np.exp(z).sum()
    out = policy(W, s)
    assert out is not None and np.allclose(out, expected), f"policy: expected {expected.round(4).tolist()}, got {out}."
    big = policy(np.array([[1000.0, 0, 0, 0], [0, 0, 0, 0]]), np.array([1.0, 0, 0, 0]))
    assert np.all(np.isfinite(big)) and np.isclose(big[0], 1.0), "policy overflows: subtract the maximum of the logits."
    for a in (0, 1):
        g = grad_log_policy(W, s, a)
        assert np.shape(g) == (2, 4), f"grad_log_policy must return a 2 x 4 array, like W: got shape {np.shape(g)}."
        numeric = np.zeros((2, 4))
        for i in range(2):
            for j in range(4):
                E = np.zeros((2, 4))
                E[i, j] = 1e-6
                numeric[i, j] = (np.log(policy(W + E, s)[a]) - np.log(policy(W - E, s)[a])) / 2e-6
        assert np.allclose(g, numeric, atol=1e-5), (
            f"grad_log_policy(W, s, {a}) differs from the finite differences: expected (e_a - pi) s^T = {numeric.round(4).tolist()}."
        )
    _ok("policy and grad_log_policy")


def check_returns_to_go(returns_to_go):
    out = returns_to_go(np.array([1.0, 0.0, 2.0]), 0.5)
    assert out is not None and np.allclose(out, [1.5, 1.0, 2.0]), (
        f"Rewards (1, 0, 2), gamma = 1/2: G = (1 + 0 + 2/4, 0 + 2/2, 2) = (1.5, 1, 2). Got {out}."
    )
    assert np.allclose(returns_to_go(np.ones(4), 1.0), [4, 3, 2, 1]), "With gamma = 1, G_t is the sum of the remaining rewards."
    _ok("returns_to_go")


def check_reinforce_gradient(reinforce_gradient):
    W = np.zeros((2, 4))
    states = np.array([[1.0, 0, 0, 0], [0, 1.0, 0, 0]])
    actions = np.array([1, 0])
    rewards = np.array([1.0, 1.0])
    # G = (1.5, 1); grad log pi at W = 0 is (e_a - 1/2) s^T
    expected = np.zeros((2, 4))
    expected[:, 0] = 1.5 * np.array([-0.5, 0.5])
    expected[:, 1] = 1.0 * np.array([0.5, -0.5])
    out = reinforce_gradient(W, states, actions, rewards, 0.5)
    assert out is not None and np.allclose(out, expected), (
        f"Two steps, rewards (1, 1), gamma = 1/2, W = 0: expected {expected.tolist()}, got {np.round(out, 3).tolist() if out is not None else None}."
    )
    out = reinforce_gradient(W, states, actions, rewards, 0.5, 1.0)
    expected[:, 0] = 0.5 * np.array([-0.5, 0.5])
    expected[:, 1] = 0.0
    assert np.allclose(out, expected), "With a baseline b, each G_t is replaced with G_t - b."
    _ok("reinforce_gradient")


def check_actor_critic(ActorCritic):
    from .control import quadratic_features

    agent = ActorCritic(alpha_w=0.1, alpha_v=0.5, gamma=0.9, rng=0)
    assert np.shape(agent.W) == (2, 4) and np.shape(agent.v) == (15,), "W must be 2 x 4 and v of length 15, both zero at the start."
    s = np.array([0.0, 0.0, 0.1, 0.0])
    s_next = np.array([0.0, 0.0, 0.0, 0.0])
    agent.v = quadratic_features(s_next) * 0 + np.eye(15)[0]  # V(s) = 1 for any s
    agent.update(s, 1, 1.0, s_next, False)
    delta = 1.0 + 0.9 - 1.0
    psi = quadratic_features(s)
    assert np.allclose(agent.v, np.eye(15)[0] + 0.5 * delta * psi), "Critic: v + alpha_v delta psi(s), with delta = r + gamma V(s') - V(s)."
    expected_W = 0.1 * delta * np.outer(np.array([-0.5, 0.5]), s)
    assert np.allclose(agent.W, expected_W), "Actor: W + alpha_w delta grad log pi_W(a | s), with the delta computed before updating v."
    agent = ActorCritic(alpha_w=0.1, alpha_v=0.5, gamma=0.9, rng=0)
    agent.v = np.eye(15)[0].copy()
    agent.update(s, 0, 1.0, s_next, True)
    assert np.allclose(agent.v, np.eye(15)[0]), "When the pole falls (failed = True), the target is r alone: here delta = 1 - 1 = 0."
    _ok("ActorCritic")


# ---------------------------------------------------------------------------
# Lab 9

def _small_logits():
    rng = np.random.default_rng(0)
    return rng.normal(size=(27, 27))


def check_log_prob(log_prob):
    from .language import encode, softmax

    L = _small_logits()
    P = softmax(L)
    word = encode("ab")
    expected = np.log(P[26, 0]) + np.log(P[0, 1]) + np.log(P[1, 26])
    out = log_prob(L, word)
    assert out is not None and np.isclose(out, expected), (
        f"log pi('ab') = log pi(a | start) + log pi(b | a) + log pi(end | b) = {expected:.4f}, got {out}."
    )
    padded = np.concatenate([word, [-1, -1]])
    assert np.isclose(log_prob(L, padded), expected), "The padding tokens -1 after the end must be ignored."
    _ok("log_prob")


def check_reward_model(fit_reward_model):
    from .language import encode

    chosen = np.array([encode("aa"), encode("ab")], dtype=object)
    rejected = np.array([encode("zz"), encode("zb")], dtype=object)
    w = fit_reward_model(chosen, rejected, n_iter=1, lr=1.0)
    # delta = (2 e_a - 2 e_z, e_a - e_z), w = 0: gradient = 1/2 * sum of the deltas / 2
    assert w is not None and w.shape == (28,), "fit_reward_model must return a vector of 28 weights."
    expected = np.zeros(28)
    expected[0], expected[25] = 0.75, -0.75
    assert np.allclose(w, expected), (
        "One step from w = 0 with lr = 1: sigma(0) = 1/2, so w = mean of the Delta_i / 2: "
        f"0.75 for 'a', -0.75 for 'z', 0 elsewhere. Got {np.round(w, 3).tolist()}."
    )
    w = fit_reward_model(chosen, rejected, n_iter=200, lr=1.0)
    assert w[0] > 1 and w[25] < -1, "After many steps, w must favor 'a' and penalize 'z'."
    _ok("fit_reward_model")


def check_grad_log_prob(grad_log_prob):
    from .language import encode, softmax

    L = _small_logits()
    word = encode("aba")
    g = grad_log_prob(L, word)
    assert g is not None and np.shape(g) == (27, 27), "grad_log_prob must return a 27 x 27 array."
    eps = 1e-6
    for p, c in [(26, 0), (0, 1), (1, 0), (0, 26), (0, 3), (5, 5)]:
        E = np.zeros((27, 27))
        E[p, c] = eps
        num = (np.sum(np.log(_pick(softmax(L + E), word))) - np.sum(np.log(_pick(softmax(L - E), word)))) / (2 * eps)
        assert np.isclose(g[p, c], num, atol=1e-5), (
            f"Entry ({p}, {c}) of grad_log_prob differs from the finite differences ({num:.4f}, got {g[p, c]:.4f})."
        )
    _ok("grad_log_prob")


def _pick(P, word):
    probs, previous = [], 26
    for t in word:
        probs.append(P[previous, t])
        if t == 26:
            break
        previous = t
    return np.array(probs)


def check_rlhf_gradient(rlhf_gradient):
    from .language import encode, reference_logits, softmax

    L = reference_logits()
    words = [encode("ab"), encode("ba")]
    rewards = np.array([1.0, 0.0])
    g = rlhf_gradient(L, words, rewards, 0.0)
    P = softmax(L)

    def grad(word):
        N = np.zeros((27, 27))
        previous = 26
        for t in word:
            N[previous, t] += 1
            if t == 26:
                break
            previous = t
        return N - N.sum(axis=1, keepdims=True) * P

    expected = (0.5 * grad(words[0]) - 0.5 * grad(words[1])) / 2
    assert g is not None and np.allclose(g, expected), (
        "With beta = 0, L = L_ref, rewards (1, 0): the advantages are (1/2, -1/2), and the gradient is "
        "(1/2 grad log pi(ab) - 1/2 grad log pi(ba)) / 2."
    )
    g = rlhf_gradient(L, words, np.array([3.0, 3.0]), 1.0)
    assert np.allclose(g, 0), "At L = L_ref the KL term is zero, and equal rewards give a zero gradient (baseline)."
    L2 = L.copy()
    L2[26, 0] += 1.0  # makes 'ab' more likely than under the reference
    g = rlhf_gradient(L2, words, np.array([0.0, 0.0]), 1.0)
    assert (g * grad(words[0])).sum() < 0, "The KL penalty must push down the words that are more likely than under the reference."
    _ok("rlhf_gradient")


def check_dpo_gradient(dpo_gradient):
    from .language import encode, reference_logits, softmax

    L = reference_logits()
    chosen, rejected = [encode("ab")], [encode("ba")]
    g = dpo_gradient(L, chosen, rejected, 0.5)
    P = softmax(L)

    def grad(word):
        N = np.zeros((27, 27))
        previous = 26
        for t in word:
            N[previous, t] += 1
            if t == 26:
                break
            previous = t
        return N - N.sum(axis=1, keepdims=True) * P

    expected = -0.5 * 0.5 * (grad(chosen[0]) - grad(rejected[0]))
    assert g is not None and np.allclose(g, expected), (
        "At L = L_ref, h = 0 for both words and sigma(0) = 1/2: the gradient is -beta/2 (grad log pi(chosen) - grad log pi(rejected))."
    )
    _ok("dpo_gradient")

In [ ]:
%%writefile seqlearn/challenge.py
"""Bandit challenge (Lab 4): three tracks, one agent, a score relative to UCB1.

An agent is built by a function `make_agent(n_arms, dim, horizon, rng)` and follows the
interface of `LinearAgent`: `act(features)` returns the arm to play and
`update(x, arm, reward)` receives the result. The three tracks are linear bandits with
Bernoulli rewards:

  slots    K = 20 arms with small means (click rates), features e_1, ..., e_K
  catalog  K = 300 items described by d = 8 features, the same at every round
  users    K = 6 items and a user profile over 4 moods at each round, d = 24

The score of a track is the mean final pseudo-regret of the agent divided by the one of
UCB1, which ignores the features, on the same instances, in percent. The score of the
challenge is the mean of the three track scores: the lower, the better.
"""

import importlib.util
import time
from pathlib import Path

import numpy as np

from .agents import UCBAgent
from .linear import CatalogBandit, ContextFree, RecommendationBandit, make_catalog, run_linear

HORIZON = 5000
TRACKS = ("slots", "catalog", "users")


def make_env(track, rng):
    """Draw an instance of `track` (its parameters and its randomness) from `rng`."""
    rng = np.random.default_rng(rng)
    if track == "slots":
        n_arms = 20
        return CatalogBandit(np.eye(n_arms), rng.beta(1.5, 12, size=n_arms), rng)
    if track == "catalog":
        items, theta = make_catalog(300, 8, rng)
        return CatalogBandit(items, theta, rng)
    if track == "users":
        return RecommendationBandit(rng.beta(0.7, 2.0, size=(6, 4)), concentration=0.3, rng=rng)
    raise ValueError(f"unknown track {track!r}, expected one of {TRACKS}")


def make_reference(n_arms, dim, horizon, rng):
    """UCB1 on unrelated arms, the reference of the scores."""
    return ContextFree(UCBAgent(n_arms, 2.0, rng))


def final_regrets(make_agent, track, n_runs, seed):
    """Final pseudo-regret of `n_runs` runs on `track`, on instances derived from `seed`."""
    regrets = np.empty(n_runs)
    for i, child in enumerate(np.random.SeedSequence([seed, TRACKS.index(track)]).spawn(n_runs)):
        env_seed, agent_seed = child.spawn(2)
        env = make_env(track, np.random.default_rng(env_seed))
        dim = env.features().shape[1]
        agent = make_agent(env.n_arms, dim, HORIZON, np.random.default_rng(agent_seed))
        regrets[i] = run_linear(env, agent, HORIZON)[2].sum()
    return regrets


def evaluate(make_agent, n_runs=10, seed=0, verbose=True):
    """Evaluate an agent on the three tracks, against the reference.

    Return a dictionary: for each track, the mean final regret of the agent and of the
    reference, and the score; and the overall score under the key "score".
    """
    results = {}
    for track in TRACKS:
        start = time.perf_counter()
        agent = final_regrets(make_agent, track, n_runs, seed)
        elapsed = time.perf_counter() - start
        reference = final_regrets(make_reference, track, n_runs, seed)
        score = 100 * agent.mean() / reference.mean()
        results[track] = {"regret": agent.mean(), "reference": reference.mean(), "score": score,
                          "seconds": elapsed}
        if verbose:
            print(f"{track:8s} regret {agent.mean():7.1f} (UCB1 {reference.mean():6.1f})"
                  f"  score {score:5.1f}  [{elapsed:.0f} s]")
    results["score"] = float(np.mean([results[track]["score"] for track in TRACKS]))
    if verbose:
        print(f"score of the challenge: {results['score']:.1f} (UCB1: 100, lower is better)")
    return results


def load_agent(path):
    """Load the function `make_agent` of the file `path`, reloading the file at each call."""
    path = Path(path)
    spec = importlib.util.spec_from_file_location(f"challenge_{path.stem}", path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module.make_agent

In [ ]:
%%writefile seqlearn/__init__.py
"""Lab framework for the Sequential Decision course (ESIEE Paris)."""

# Date of the last change of the package; bump it when any module changes (make_lab_kit.py checks it).
__version__ = "2026.11.03"

from .bandits import (
    Agent,
    Bandit,
    BernoulliBandit,
    GaussianBandit,
    UniformAgent,
    pseudo_regret,
    run,
    run_many,
)
from .agents import UCBAgent
from .plotting import plot_mean

In [ ]:
import copy

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn

from seqlearn.blocks import Blocks
from seqlearn.control import CartPole
from seqlearn.plotting import plot_mean, set_style

set_style()
torch.set_num_threads(1)  # small networks: one thread is faster

## 1. Tensors

A tensor is a numpy array that can live on a GPU and remember how it was computed. Most
numpy functions have the same name: `torch.zeros`, `torch.exp`, `x.sum(dim=1)` (instead of
`axis=1`), `x @ y`. `torch.from_numpy(a)` and `x.numpy()` convert, without copying. The
default type is `float32`, and two tensors in an operation must have the same type.

In [ ]:
x = torch.tensor([[1.0, 2.0, 3.0], [0.0, -1.0, 1.0]])
print(x.shape, x.dtype)
print(x.sum(dim=1), x.max(dim=1).values, x.T @ x)

**Question 1.** Write `softmax_rows(z)`, the softmax of each row of a 2-d tensor, with
`torch.exp` and `sum` only (subtract the maximum of each row first), and compare with
`torch.softmax(z, dim=1)`.

In [ ]:
def softmax_rows(z):
    ...  # TODO


z = torch.randn(4, 3) * 10
print(softmax_rows(z))
print("same as torch.softmax:", torch.allclose(softmax_rows(z), torch.softmax(z, dim=1)))

## 2. Automatic differentiation

A tensor created with `requires_grad=True` records the operations applied to it. Calling
`.backward()` on a scalar computed from it fills its attribute `.grad` with the gradient.
This is how the gradients of networks are computed: nobody differentiates by hand.

**Question 2.** For a random $2 \times 4$ matrix $W$, a random state $s$ and $a = 1$,
compute $\log \pi_W(a \mid s)$ with `torch.log_softmax`, call `backward`, and compare
`W.grad` with the formula $(e_a - \pi_W(\cdot \mid s))\, s^\top$ of Lab 8.

In [ ]:
torch.manual_seed(0)
W = torch.randn(2, 4, requires_grad=True)
s = torch.randn(4)
a = 1
...  # TODO

Gradients add up: a second `backward` adds to `W.grad`, which must be reset to zero
between two steps. `with torch.no_grad():` turns the recording off, for computations that
are not differentiated (updates, evaluation).

## 3. Modules and optimizers

A network is a module: `nn.Linear(d, k)` computes $x \mapsto Ax + b$, `nn.Sequential`
chains modules, and `net.parameters()` lists the tensors to learn. An optimizer updates
them from their `.grad`. Every training loop has the same four lines:

```python
loss = ...              # a scalar computed from the parameters
optimizer.zero_grad()   # reset the gradients
loss.backward()         # compute them
optimizer.step()        # update the parameters
```

**Question 3.** Complete the training loop of a logistic regression on the two clouds of
points below, with the loss `nn.BCEWithLogitsLoss()` and `torch.optim.SGD` (step $0.1$),
for $200$ steps on the whole data set. Print the accuracy at the end.

In [ ]:
rng = np.random.default_rng(0)
X = torch.from_numpy(np.vstack([rng.normal([-1, 0], 1, (200, 2)), rng.normal([1, 1], 1, (200, 2))]).astype(np.float32))
y = torch.cat([torch.zeros(200), torch.ones(200)])

model = nn.Linear(2, 1)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
for step in range(200):
    ...  # TODO
with torch.no_grad():
    accuracy = ((model(X).squeeze(1) > 0).float() == y).float().mean()
print(f"loss {loss.item():.3f}, accuracy {accuracy.item():.3f}")

## 4. REINFORCE with a neural policy

The policy is now a network with one hidden layer, which outputs the two logits. As in the
lecture, REINFORCE minimizes the surrogate loss
$$L = -\sum_t (G_t - b)\, \log \pi(A_t \mid S_t) \enspace,$$
whose gradient is minus the REINFORCE estimate. Here $b$ is the mean of the $G_t$ of the
episode.

**Question 4.** Complete `reinforce_loss(policy, states, actions, G)`, where `states` is a
tensor of shape $(T, 4)$, `actions` a tensor of integers of shape $(T,)$ and `G` the
returns-to-go. Then train $3$ policies for $300$ episodes with Adam (step $0.01$), one
update per episode, and plot the mean length of the episodes.

In [ ]:
def reinforce_loss(policy, states, actions, G):
    ...  # TODO


def returns_to_go(rewards, gamma):
    G, acc = np.zeros(len(rewards), dtype=np.float32), 0.0
    for t in range(len(rewards) - 1, -1, -1):
        acc = rewards[t] + gamma * acc
        G[t] = acc
    return G


def train_reinforce(seed, n_episodes=300, gamma=0.99, lr=0.01):
    torch.manual_seed(seed)
    rng, env = np.random.default_rng(seed), CartPole(rng=seed)
    policy = nn.Sequential(nn.Linear(4, 32), nn.Tanh(), nn.Linear(32, 2))
    optimizer = torch.optim.Adam(policy.parameters(), lr=lr)
    lengths = []
    for _ in range(n_episodes):
        states, actions, rewards = [], [], []
        s, done = env.reset(), False
        while not done:
            with torch.no_grad():
                p = torch.softmax(policy(torch.as_tensor(s, dtype=torch.float32)), dim=0).numpy()
            a = int(rng.choice(2, p=p / p.sum()))
            states.append(s)
            actions.append(a)
            s, r, done = env.step(a)
            rewards.append(r)
        loss = reinforce_loss(policy, torch.as_tensor(np.array(states), dtype=torch.float32),
                              torch.as_tensor(actions), torch.from_numpy(returns_to_go(rewards, gamma)))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        lengths.append(len(rewards))
    return np.array(lengths)


...  # TODO

## 5. A deep Q-network on the falling blocks

In Lab 8, $Q_w(s, a)$ was a linear function of features chosen by hand. Here
$Q(s, a)$ is a network that reads the raw board obtained after the placement ($60$ cells,
$0$ or $1$) and the number of rows removed: it must find the useful features itself. As in
the lecture, two devices make the semi-gradient updates stable:

- a **replay buffer** keeps the last $20000$ transitions, and each update uses a
  mini-batch of $32$ transitions drawn at random among them;
- a **target network**, a copy of the network updated every $500$ steps, computes the
  targets $y = r + \gamma \max_{a'} Q^-(s', a')$.

`inputs(game, board, piece)` returns the placements, the inputs of the network for each of
them, their rewards and the boolean array of the placements that end the game, as
`placement_features` in Lab 8.

In [ ]:
def inputs(game, board, piece):
    actions = game.actions(piece)
    X = np.zeros((len(actions), game.height * game.width + 1), dtype=np.float32)
    rewards = np.zeros(len(actions), dtype=np.float32)
    lost = np.zeros(len(actions), dtype=bool)
    for k, action in enumerate(actions):
        dropped = game.drop(board, piece, action)
        if dropped is None:
            lost[k], rewards[k] = True, game.lose
        else:
            new, cleared = game.clear(dropped)
            X[k, :-1], X[k, -1], rewards[k] = new.ravel(), cleared, cleared
    return actions, X, rewards, lost


def q_values(net, X, rewards, lost):
    """Q-values of the placements: the network, or the reward for those that end the game."""
    with torch.no_grad():
        q = net(torch.from_numpy(X)).squeeze(1).numpy()
    return np.where(lost, rewards, q)

**Question 5.** Complete `dqn_loss(net, target_net, batch, gamma)`. A transition of the
buffer is `(x, r, next)`, where `x` is the input of the placement played, `r` its reward,
and `next` is `None` if the game ended, and otherwise the triple `(X, rewards, lost)` of the
placements of the next state. The loss is the mean of $(Q(x) - y)^2$ over the batch, where
$y$ is computed with the target network and is not differentiated.

In [ ]:
def dqn_loss(net, target_net, batch, gamma):
    x = torch.from_numpy(np.array([t[0] for t in batch]))
    ...  # TODO


def train_dqn(seed, n_games=300, gamma=0.95, epsilon=0.05, lr=1e-3, batch_size=32, sync=500):
    torch.manual_seed(seed)
    rng, game = np.random.default_rng(seed), Blocks(max_pieces=200, rng=seed)
    net = nn.Sequential(nn.Linear(61, 64), nn.ReLU(), nn.Linear(64, 1))
    target_net = copy.deepcopy(net)
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    buffer, rows, step = [], [], 0
    for _ in range(n_games):
        (board, piece), done = game.reset(), False
        actions, X, rewards, lost = inputs(game, board, piece)
        while not done:
            q = q_values(net, X, rewards, lost)
            k = int(rng.integers(len(actions))) if rng.random() < epsilon else int(np.argmax(q))
            (board, piece), r, done = game.step(actions[k])
            nxt = None if done else inputs(game, board, piece)
            if not lost[k]:
                buffer.append((X[k], r, None if done else nxt[1:]))
                if len(buffer) > 20000:
                    buffer.pop(0)
            if not done:
                actions, X, rewards, lost = nxt
            step += 1
            if len(buffer) >= 500:
                batch = [buffer[i] for i in rng.integers(len(buffer), size=batch_size)]
                loss = dqn_loss(net, target_net, batch, gamma)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            if step % sync == 0:
                target_net.load_state_dict(net.state_dict())
        rows.append(game.rows)
    return np.array(rows)

**Question 6.** Train $2$ agents for $300$ games (about $2$ minutes each) and plot the mean
number of rows per game. Compare with linear Q-learning in Lab 8, which reached about $70$
rows after $200$ games.

In [ ]:
...  # TODO